# Anchor-scarce GNN localisation — all-in-one Colab notebook (E(2)-Equivariant & Bug-Fixed)

Everything is inside this notebook: no files to upload.


In [ ]:
# GPU check & Multi-GPU detection
import torch
assert torch.cuda.is_available(), "No GPU found! In Colab: Runtime > Change runtime type > GPU. In Kaggle: Settings > Accelerator > GPU T4 x2."
num_gpus = torch.cuda.device_count()
print(f"GPUs available: {num_gpus}")
for i in range(num_gpus):
    vram_gib = torch.cuda.get_device_properties(i).total_memory / (1024**3)
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)} ({vram_gib:.1f} GiB VRAM)")
if num_gpus >= 2:
    print("-> Multi-GPU acceleration active: training and evaluation will leverage BOTH GPUs simultaneously!")
!pip -q install cvxpy


In [ ]:
# ---------------- settings ----------------
MODE          = "quick"   # "quick" | "short" | "medium" | "full"
USE_DRIVE     = False     # True: keep everything in Google Drive (survives disconnects, Colab only)
RUN_ABLATIONS = True      # uniform / huber / encoder_only models
RUN_TRANSFER  = True      # 7 models: train on 1, 2 or all 3 graph types, test on all
RUN_SDP       = True      # SDP baseline (slowest one)
TEST_GRAPHS   = 50        # test graphs per result cell (any mode); None = the mode's default
# ------------------------------------------
import os
os.environ["PYTHONUNBUFFERED"] = "1"
IS_KAGGLE = os.path.exists("/kaggle/working")
if IS_KAGGLE:
    WORKDIR = "/kaggle/working/gnn_loc"
else:
    WORKDIR = "/content/drive/MyDrive/gnn_loc" if USE_DRIVE else "/content/gnn_loc"
    if USE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
os.makedirs(WORKDIR, exist_ok=True)
%cd {WORKDIR}

for k in ["QUICK", "N_TRAIN", "N_VAL", "EPOCHS", "TEST_PER_LEVEL", "BA_PER_M", "PERGRAPH_EPOCHS"]:
    os.environ.pop(k, None)
if MODE == "quick":
    os.environ["QUICK"] = "1"
elif MODE == "short":     # ~45 min on a T4: main model + ablations, all methods incl. SDP
    os.environ.update(N_TRAIN="1000", N_VAL="200", EPOCHS="20", TEST_PER_LEVEL="5", PERGRAPH_EPOCHS="300")
    RUN_TRANSFER = False     # 7 more models do not fit in 45 min
elif MODE == "medium":
    os.environ.update(N_TRAIN="1000", N_VAL="200", EPOCHS="20", TEST_PER_LEVEL="10", PERGRAPH_EPOCHS="300")
elif MODE != "full":
    raise ValueError(MODE)
if TEST_GRAPHS and MODE != "quick":              # same number of graphs behind every number: main, n=100/200 and BA tests
    os.environ.update(TEST_PER_LEVEL=str(TEST_GRAPHS), BA_PER_M=str(TEST_GRAPHS))
os.environ["RUN_SDP"] = "1" if RUN_SDP else "0"
os.environ["RUN_TRANSFER"] = "1" if RUN_TRANSFER else "0"
import subprocess, sys
def sh(cmd, **env):
    # run a command, stream its output, stop the notebook if it fails
    print(f"$ {cmd}", flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                         env={**os.environ, **env})
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait() != 0:
        raise RuntimeError(f"failed (exit code {p.returncode}): {cmd}")
print("MODE:", MODE, "| test graphs per cell:", os.environ.get("TEST_PER_LEVEL", "mode default"), "| working dir:", WORKDIR)


## The code
Each cell writes one file. These are exactly the files in the repo.

In [ ]:
%%writefile config.py
"""
config.py -- every experiment setting in ONE place.
Change things here, never inside the step files.
"""
import os

# ----------------------------------------------------------------------------
# Problem size
# ----------------------------------------------------------------------------
N_NODES        = 50          # nodes per test graph (the paper's main setting)
N_ANCHORS      = 3           # the theoretical minimum in 2-D (d + 1)
COORD_RANGE    = 10.0        # points are uniform in [-COORD_RANGE, COORD_RANGE]^2

# ----------------------------------------------------------------------------
# How edges go missing
#   "random" : spanning tree + random extra edges (fast, always connected)
#   "rgg"    : random geometric graph -- keep the SHORTEST pairs
#              (+ Euclidean MST so it is connected). This is how real
#              sensor networks lose links (radio range).
#   "ba"     : Barabasi-Albert, m chosen so m*(n-m) edges match the missing %.
#              BA has at most n^2/4 edges, so it only exists at >= ~50 %
#              missing; lower test levels are skipped for BA.
# ----------------------------------------------------------------------------
GRAPH_TYPES     = ["random", "rgg", "ba"]      # training mix
MISSING_LEVELS  = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]

# ----------------------------------------------------------------------------
# Noise on the RAW distances (not squared):
#   d_obs = d_true * (1 + sigma * N(0,1))  +  Bernoulli(p_nlos) * U[0, nlos_max]
# The second term is a POSITIVE non-line-of-sight (NLOS) bias, as in
# Yan et al. (ICASSP 2021). nlos_max is in coordinate units (area is 20 x 20).
# ----------------------------------------------------------------------------
NOISE_MODELS = {
    "clean": dict(sigma=0.00, p_nlos=0.0, nlos_max=0.0),
    "gauss": dict(sigma=0.05, p_nlos=0.0, nlos_max=0.0),
    "nlos":  dict(sigma=0.02, p_nlos=0.2, nlos_max=4.0),
}

# ----------------------------------------------------------------------------
# Datasets (step 1)
# ----------------------------------------------------------------------------
N_TRAIN          = 3000      # training graphs (random missing %, noise, type)
N_VAL            = 400
TRAIN_N_RANGE    = (30, 80)  # train on VARIABLE sizes -> generalises in n
TRAIN_MISSING    = (0.10, 0.90)
TEST_PER_LEVEL   = 20        # test graphs per (scenario, missing level)
TEST_SCENARIOS   = [("random", "clean"), ("random", "nlos"),
                    ("rgg", "clean"),    ("rgg", "nlos"),
                    ("ba", "clean"),     ("ba", "nlos")]
GENERALISE_SIZES = [100, 200]          # extra test sets, never seen in training
GENERALISE_SCENARIO = ("rgg", "nlos")
GENERALISE_LEVELS = [0.5, 0.7, 0.9]
BA_M_VALUES      = [1, 2, 3, 4, 5]     # rigidity-threshold experiment
BA_PER_M         = 20

# Cross-graph-type transfer: train on a subset of graph types, test on all.
# Every training set has the SAME total size (N_TRAIN, split equally over its
# types), so differences come from variety, not from amount of data.
# Validation (checkpoint selection) uses only the training types: no peeking
# at the held-out type.
RUN_TRANSFER        = True
TRANSFER_TRAIN_SETS = ["random", "rgg", "ba",                   # train on 1, test on the other 2
                       "random+rgg", "random+ba", "rgg+ba",     # train on 2, test on the 3rd
                       "all"]                                   # train on all 3
TRANSFER_MIN_MISSING = 0.5   # matrix averages over missing >= 50 % (BA exists only there)
SEED             = 1234

# ----------------------------------------------------------------------------
# Model (step 2)
# ----------------------------------------------------------------------------
HIDDEN          = 96
ENC_LAYERS      = 5          # message-passing layers in the encoder
REFINE_STEPS    = 10         # unrolled learned-SMACOF steps during TRAINING
TEST_REFINE_STEPS = 50       # steps at TEST time (same shared step, run longer:
                             # exact fits are fixed points, so more steps only converge further)
VARIANT         = "learned"  # "learned" | "uniform" | "huber" | "encoder_only"
LOSS_MODE       = "supervised"   # "supervised" (synthetic GT) | "selfsup" (no GT)
HUBER_DELTA     = 0.05       # in normalised units (anchor-triangle scale = 1)
AUX_WEIGHT      = 0.3        # weight of the encoder-output loss (deep supervision)
EPOCHS          = 50
BATCH_GRAPHS    = 32
LR              = 1e-3
WEIGHT_DECAY    = 1e-5
GRAD_CLIP       = 1.0

# ----------------------------------------------------------------------------
# Evaluation (step 3)
# ----------------------------------------------------------------------------
RUN_SDP            = True    # Biswas-Ye SDP via cvxpy (slowest baseline)
RUN_PERGRAPH_SAGE  = True    # your previous approach (per-graph training)
RUN_MPNN_LOC       = True    # MPNN-Loc (Xu & Lin, IFAC 2025)
PERGRAPH_EPOCHS    = 1000
YAN_GCN_EPOCHS     = 200
MPNN_LOC_EPOCHS    = 200
RUN_GENERALISATION = True
RUN_BA_EXPERIMENT  = True


def train_types(key):
    """'rgg+ba' -> ['rgg', 'ba'];  'all' -> GRAPH_TYPES."""
    types = list(GRAPH_TYPES) if key == "all" else key.split("+")
    bad = [t for t in types if t not in GRAPH_TYPES]
    if bad:
        raise ValueError(f"unknown graph type(s) {bad}; choose from {GRAPH_TYPES} or 'all'")
    return types

# ----------------------------------------------------------------------------
# Paths
# ----------------------------------------------------------------------------
ROOT        = os.path.dirname(os.path.abspath(__file__))
DATA_DIR    = os.path.join(ROOT, "data")
CKPT_DIR    = os.path.join(ROOT, "checkpoints")
RESULTS_DIR = os.path.join(ROOT, "results")
for _d in (DATA_DIR, CKPT_DIR, RESULTS_DIR):
    os.makedirs(_d, exist_ok=True)

# Any of these can be overridden from the environment, e.g.  EPOCHS=10 python step2_train.py
for _k in ("N_TRAIN", "N_VAL", "TEST_PER_LEVEL", "BA_PER_M", "EPOCHS", "PERGRAPH_EPOCHS", "REFINE_STEPS",
           "TEST_REFINE_STEPS", "MPNN_LOC_EPOCHS"):
    if _k in os.environ:
        globals()[_k] = int(os.environ[_k])
for _k in ("RUN_SDP", "RUN_PERGRAPH_SAGE", "RUN_TRANSFER", "RUN_MPNN_LOC"):       # e.g.  RUN_SDP=0 python step3_evaluate.py
    if _k in os.environ:
        globals()[_k] = os.environ[_k] == "1"
SDP_MAX_N = 100   # SDP gets slow for big graphs (n=200 ~ 1 min/graph): skipped above this

# Set QUICK=1 in the environment for a tiny smoke-test run.
if os.environ.get("QUICK") == "1":
    N_TRAIN, N_VAL, TEST_PER_LEVEL, EPOCHS = 200, 40, 2, 3
    MISSING_LEVELS = [0.3, 0.7, 0.9]
    GENERALISE_LEVELS = [0.7]
    BA_PER_M = 2
    PERGRAPH_EPOCHS, YAN_GCN_EPOCHS, MPNN_LOC_EPOCHS = 50, 30, 30


In [ ]:
%%writefile common.py
"""
common.py -- shared building blocks used by step1 / step2 / step3.

    1. geometry           : points, distance matrices, E(2) canonical anchor frame
    2. graph generators   : random (spanning tree + extra edges), RGG (MST-first), BA
    3. noise              : multiplicative Gaussian + positive NLOS bias
    4. anchors            : well-spread anchor selection
    5. rigidity           : is the answer even unique?  (graph + per node)
    6. features           : E(2)-equivariant features (supports any K >= 3 anchors)
    7. metrics            : coordinate RMSE, certified RMSE, relative EDM error
"""
import numpy as np
import networkx as nx
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import shortest_path, minimum_spanning_tree


# =============================================================================
# 1. Geometry & E(2) Canonical Frame
# =============================================================================
def random_points(n, rng, coord_range=10.0):
    """n points uniform in [-R, R]^2, shape (n, 2)."""
    return rng.uniform(-coord_range, coord_range, size=(n, 2))


def dist_matrix(P):
    """Raw (NOT squared) Euclidean distance matrix of points P (n, 2)."""
    diff = P[:, None, :] - P[None, :, :]
    return np.sqrt((diff ** 2).sum(-1))


def best_anchor_triangle(anchor_xy):
    """If K > 3 anchors are given, pick the 3 anchors with the largest triangle area
    to define a well-conditioned canonical coordinate frame."""
    K = len(anchor_xy)
    if K == 3:
        return np.array([0, 1, 2], dtype=np.int64)
    best_tri, best_area = (0, 1, 2), -1.0
    for a in range(K):
        for b in range(a + 1, K):
            for c in range(b + 1, K):
                u = anchor_xy[b] - anchor_xy[a]
                v = anchor_xy[c] - anchor_xy[a]
                area = abs(u[0] * v[1] - u[1] * v[0]) * 0.5
                if area > best_area:
                    best_area = area
                    best_tri = (a, b, c)
    return np.array(best_tri, dtype=np.int64)


def canonical_anchor_frame(anchor_xy):
    """
    Constructs an exact E(2)-equivariant canonical coordinate frame from K >= 3 anchors:
      1. Translation: origin = centroid c of the 3 reference anchors
      2. Scale: unit = mean pairwise distance s of the 3 reference anchors
      3. Rotation: align vector (anchor_1 - anchor_0) with positive x-axis
      4. Reflection: flip y-axis so anchor_2 lies in the upper half-plane (y >= 0)
    Returns (c, s, R, tri) where R is a (2, 2) orthogonal matrix in O(2) such that:
      X_can = ((X_orig - c) / s) @ R
      X_orig = (X_can @ R.T) * s + c
    """
    tri = best_anchor_triangle(anchor_xy)
    A3 = anchor_xy[tri].astype(np.float64)
    c = A3.mean(0)
    s = np.mean([np.linalg.norm(A3[i] - A3[j]) for i in range(3) for j in range(i + 1, 3)])
    s = max(float(s), 1e-6)
    A_s = (A3 - c) / s
    u = A_s[1] - A_s[0]
    norm_u = np.linalg.norm(u)
    e1 = u / norm_u if norm_u > 1e-9 else np.array([1.0, 0.0])
    e2 = np.array([-e1[1], e1[0]])
    if np.dot(A_s[2] - A_s[0], e2) < 0:
        e2 = -e2
    R = np.column_stack([e1, e2])
    return c.astype(np.float32), np.float32(s), R.astype(np.float32), tri


# =============================================================================
# 2. Graph generators  -- all return an (m, 2) array of undirected edges i<j
# =============================================================================
def _target_edges(n, missing):
    total = n * (n - 1) // 2
    return max(n - 1, int(round(total * (1.0 - missing))))


def graph_random(n, missing, rng):
    """
    "Spanning-tree" generator.
      1. random spanning tree: node order is shuffled and each node attaches
         to a uniformly random earlier node   -> connected, n-1 edges
      2. add uniformly random extra pairs until the target edge count.
    """
    order = rng.permutation(n)
    edges = set()
    for k in range(1, n):
        a, b = order[k], order[rng.integers(0, k)]
        edges.add((min(a, b), max(a, b)))
    target = _target_edges(n, missing)
    iu = np.array(np.triu_indices(n, 1)).T
    rng.shuffle(iu)
    for a, b in iu:
        if len(edges) >= target:
            break
        edges.add((int(a), int(b)))
    return np.array(sorted(edges), dtype=np.int64)


def graph_rgg(P, missing):
    """
    Random geometric (radio-range) graph:
      1. Start with the Euclidean minimum spanning tree (n-1 edges) so the graph
         is guaranteed connected.
      2. Add the shortest remaining pairs until the exact target edge count is reached.
    """
    n = len(P)
    D = dist_matrix(P)
    mst = minimum_spanning_tree(csr_matrix(D)).tocoo()
    edges = set()
    for a, b in zip(mst.row, mst.col):
        edges.add((int(min(a, b)), int(max(a, b))))
    target = _target_edges(n, missing)
    iu = np.triu_indices(n, 1)
    order = np.argsort(D[iu])
    for idx in order:
        if len(edges) >= target:
            break
        edges.add((int(iu[0][idx]), int(iu[1][idx])))
    return np.array(sorted(edges), dtype=np.int64)


def ba_m_for(n, missing):
    """m whose BA edge count m*(n-m) is closest to the target for this missing %."""
    target = _target_edges(n, missing)
    ms = np.arange(1, n // 2 + 1)
    return int(ms[np.argmin(np.abs(ms * (n - ms) - target))])


def ba_reachable(n, missing, tol=0.05):
    """True if a BA graph can match this missing % to within tol."""
    m = ba_m_for(n, missing)
    total = n * (n - 1) // 2
    return abs((1.0 - m * (n - m) / total) - missing) <= tol


def graph_ba(n, m, rng):
    """Barabasi-Albert graph: each new node attaches to m existing nodes."""
    G = nx.barabasi_albert_graph(n, m, seed=int(rng.integers(1 << 31)))
    e = np.array([(min(a, b), max(a, b)) for a, b in G.edges()], dtype=np.int64)
    return e


def is_connected(n, edges):
    G = nx.Graph()
    G.add_nodes_from(range(n))
    G.add_edges_from(edges.tolist())
    return nx.is_connected(G)


# =============================================================================
# 3. Noise (on RAW distances)
# =============================================================================
def noisy_distances(d_true, rng, sigma=0.0, p_nlos=0.0, nlos_max=0.0):
    """d_obs = d (1 + sigma*eps) + b * U[0, nlos_max],  b ~ Bernoulli(p_nlos)."""
    d = d_true * (1.0 + sigma * rng.standard_normal(d_true.shape))
    if p_nlos > 0:
        bias = rng.uniform(0, nlos_max, d_true.shape) * (rng.random(d_true.shape) < p_nlos)
        d = d + bias
    return np.maximum(d, 1e-3)


# =============================================================================
# 4. Anchors
# =============================================================================
def pick_anchors(P, k, rng, tries=60):
    """k anchors, chosen among `tries` random k-subsets to maximise the
    smallest triangle area (well-spread, never collinear)."""
    n = len(P)
    best, best_score = None, -1.0
    for _ in range(tries):
        idx = rng.choice(n, size=k, replace=False)
        A = P[idx]
        score = np.inf
        for a in range(k):
            for b in range(a + 1, k):
                for c in range(b + 1, k):
                    u, v = A[b] - A[a], A[c] - A[a]
                    score = min(score, abs(u[0] * v[1] - u[1] * v[0]) / 2)
        if score > best_score:
            best, best_score = idx, score
    return np.sort(best)


# =============================================================================
# 5. Rigidity: is the answer unique?
# =============================================================================
def _trilaterate_from(Adj, seed):
    """Grow a set from `seed`: add any node with >= 3 neighbours in the set."""
    done = np.zeros(len(Adj), bool)
    done[list(seed)] = True
    while True:
        new = (Adj @ done >= 3) & ~done
        if not new.any():
            return done
        done |= new


def trilateration_certified(n, edges, anchors, max_seeds=3000, rng=None):
    """Per-node SUFFICIENT certificate of unique localisability (generic positions)."""
    Adj = np.zeros((n, n), np.int32)
    Adj[edges[:, 0], edges[:, 1]] = 1
    Adj[edges[:, 1], edges[:, 0]] = 1
    anchors = list(anchors)
    cert = np.zeros(n, bool)
    comp = _trilaterate_from(Adj, anchors)
    cert |= comp
    tris = []
    for i, j in edges:
        for k in np.nonzero(Adj[i] & Adj[j])[0]:
            if k > j:
                tris.append((i, j, k))
    if rng is not None and len(tris) > max_seeds:
        tris = [tris[t] for t in rng.choice(len(tris), max_seeds, replace=False)]
    seen = np.zeros(n, bool)
    for t in tris:
        if seen[list(t)].all():
            continue
        comp = _trilaterate_from(Adj, t)
        seen |= comp
        if comp[anchors].all():
            cert |= comp
    return cert


def globally_rigid(n, edges, anchors, rng, d=2, tol=1e-8):
    """Randomised generic GLOBAL rigidity test (Gortler-Healy-Thurston 2010)."""
    e = set(map(tuple, edges.tolist()))
    for i in range(len(anchors)):
        for j in range(i + 1, len(anchors)):
            a, b = sorted((int(anchors[i]), int(anchors[j])))
            e.add((a, b))
    E = np.array(sorted(e))
    q = rng.standard_normal((n, d))
    R = np.zeros((len(E), d * n))
    for k, (i, j) in enumerate(E):
        diff = q[i] - q[j]
        R[k, d * i:d * i + d] = diff
        R[k, d * j:d * j + d] = -diff
    s = np.linalg.svd(R, compute_uv=False)
    rankR = int((s > tol * s[0]).sum())
    if rankR < d * n - d * (d + 1) // 2:
        return False
    z = rng.standard_normal(len(E))
    w = z - R @ np.linalg.lstsq(R, z, rcond=None)[0]
    Om = np.zeros((n, n))
    for k, (i, j) in enumerate(E):
        Om[i, j] -= w[k]
        Om[j, i] -= w[k]
        Om[i, i] += w[k]
        Om[j, j] += w[k]
    ev = np.abs(np.linalg.eigvalsh(Om))
    rankOm = int((ev > 1e-7 * max(ev.max(), 1e-12)).sum())
    return rankOm == n - d - 1


# =============================================================================
# 6. Features  -- built ONLY from observed distances + anchor coordinates
# =============================================================================
def geodesic_matrix(n, edges, d_obs):
    """Shortest-path lengths through the observed graph (n, n)."""
    W = csr_matrix((np.r_[d_obs, d_obs], (np.r_[edges[:, 0], edges[:, 1]],
                                         np.r_[edges[:, 1], edges[:, 0]])), shape=(n, n))
    return shortest_path(W, directed=False)


def hop_matrix(n, edges, rows):
    W = csr_matrix((np.ones(2 * len(edges)), (np.r_[edges[:, 0], edges[:, 1]],
                                              np.r_[edges[:, 1], edges[:, 0]])), shape=(n, n))
    return shortest_path(W, directed=False, unweighted=True, indices=rows)


def multilaterate(anchor_xy, g):
    """Linearised least-squares multilateration (DV-distance style)."""
    a1 = anchor_xy[0]
    A = 2.0 * (anchor_xy[1:] - a1)                                  # (K-1, 2)
    b = ((anchor_xy[1:] ** 2).sum(1) - (a1 ** 2).sum())[None, :] \
        - g[:, 1:] ** 2 + g[:, :1] ** 2                             # (n, K-1)
    X, *_ = np.linalg.lstsq(A, b.T, rcond=None)
    return X.T


def build_features(n, edges, d_obs, anchors, anchor_xy):
    """
    Returns a dict of arrays in an E(2)-CANONICALIZED frame:
        origin = anchor centroid (c), unit = mean pairwise anchor distance (s),
        orientation = canonical O(2) basis R from the 3 best-spread anchors (tri).
    Node features (19, invariant to K >= 3 anchors):
        is_anchor (1) | own anchor xy in canonical frame (2) | x0 multilateration init (2)
        geodesic dists to reference anchor triangle (3) | canonical anchor triangle xy (6)
        hop counts to reference anchor triangle / 10 (3) | log(1+degree) (1) | mean edge length (1)
    Edge features (3, per DIRECTED edge):
        d_obs | d_obs^2 | |d_obs - ||x0_i - x0_j|||   (last one flags NLOS-like edges)
    """
    c, s, R, tri = canonical_anchor_frame(anchor_xy)
    A_n = ((anchor_xy - c) / s) @ R
    d_n = d_obs / s

    G_all = geodesic_matrix(n, edges, d_n)[:, anchors]              # (n, K)
    H_all = hop_matrix(n, edges, anchors).T                         # (n, K)
    x0 = multilaterate(A_n, G_all)
    x0[anchors] = A_n

    G_feat = G_all[:, tri]
    H_feat = H_all[:, tri]
    A_tri = A_n[tri]

    deg = np.bincount(edges.ravel(), minlength=n).astype(float)
    sum_len = np.bincount(edges[:, 0], weights=d_n, minlength=n) + \
              np.bincount(edges[:, 1], weights=d_n, minlength=n)
    is_a = np.zeros(n)
    is_a[anchors] = 1
    own = np.zeros((n, 2))
    own[anchors] = A_n
    X = np.concatenate([
        is_a[:, None], own, x0, G_feat,
        np.repeat(A_tri.reshape(1, -1), n, 0),
        H_feat / 10.0, np.log1p(deg)[:, None], (sum_len / np.maximum(deg, 1))[:, None],
    ], axis=1)

    src = np.r_[edges[:, 0], edges[:, 1]]
    dst = np.r_[edges[:, 1], edges[:, 0]]
    dd = np.r_[d_n, d_n]
    incons = np.abs(dd - np.linalg.norm(x0[src] - x0[dst], axis=1))
    EF = np.stack([dd, dd ** 2, incons], 1)
    return dict(x=X.astype(np.float32), x0=x0.astype(np.float32),
                edge_index=np.stack([src, dst]).astype(np.int64),
                edge_attr=EF.astype(np.float32), d_edge=dd.astype(np.float32),
                center=c.astype(np.float32), scale=np.float32(s),
                rot=R.astype(np.float32))


# =============================================================================
# One complete graph sample
# =============================================================================
def make_graph(rng, n, graph_type, missing, noise, n_anchors=3, coord_range=10.0,
               ba_m=None, check_global=True):
    """
    Build one sample. Everything a method may use lives under 'obs';
    ground truth lives under 'true_xy' (training loss + scoring only).
    """
    P = random_points(n, rng, coord_range)
    if graph_type == "random":
        edges = graph_random(n, missing, rng)
    elif graph_type == "rgg":
        edges = graph_rgg(P, missing)
    elif graph_type == "ba":
        if ba_m is None:                  # main comparison: m chosen from the missing %
            ba_m = ba_m_for(n, missing)
        edges = graph_ba(n, ba_m, rng)
    else:
        raise ValueError(graph_type)
    d_true = np.linalg.norm(P[edges[:, 0]] - P[edges[:, 1]], axis=1)
    d_obs = noisy_distances(d_true, rng, **noise)
    anchors = pick_anchors(P, n_anchors, rng)
    feats = build_features(n, edges, d_obs, anchors, P[anchors])
    total = n * (n - 1) // 2
    gr = globally_rigid(n, edges, anchors, rng) if check_global else None
    cert = np.ones(n, bool) if gr else trilateration_certified(n, edges, anchors, rng=rng)
    return dict(
        n=n, graph_type=graph_type, missing_nominal=float(missing),
        missing_actual=1.0 - len(edges) / total,
        edges=edges, d_obs=d_obs.astype(np.float32),
        anchors=anchors, anchor_xy=P[anchors].astype(np.float32),
        true_xy=P.astype(np.float32),
        certified=cert,            # nodes whose position is provably unique
        globally_rigid=gr,         # whole graph uniquely localisable?
        ba_m=ba_m,                 # None unless graph_type == "ba"
        **feats,
    )


# =============================================================================
# 7. Metrics (ground truth used HERE only)
# =============================================================================
def unknown_mask(g):
    m = np.ones(g["n"], bool)
    m[g["anchors"]] = False
    return m


def coord_rmse(g, X, mask=None):
    """RMSE over non-anchor nodes (or a sub-mask of them)."""
    m = unknown_mask(g) if mask is None else (mask & unknown_mask(g))
    if m.sum() == 0:
        return np.nan
    return float(np.sqrt(((X[m] - g["true_xy"][m]) ** 2).sum(1).mean()))


def rel_edm_error(g, X):
    """||D_hat - D||_F / ||D||_F on raw distances (blind to reflections)."""
    D, Dh = dist_matrix(g["true_xy"]), dist_matrix(X)
    return float(np.linalg.norm(D - Dh) / np.linalg.norm(D))


In [ ]:
%%writefile model.py
"""
model.py -- the proposed network:  AnchorGNN encoder  +  learned unrolled SMACOF.

Pure PyTorch (no torch_geometric needed): message passing uses index_add_.

    features --> [ENCODER: L edge-aware MPNN layers] --> h_i, e_ij, x_A
                                                          |
                 [REFINER: T unrolled learned-SMACOF steps, anchors pinned]
                                                          v
                                                       x_T  (prediction)

Variants (for the ablation):
    "learned"      : weights w_ij from an MLP           (proposed)
    "uniform"      : w_ij = 1   -> exactly Jacobi dwMDS / SMACOF, no learning
    "huber"        : w_ij = min(1, delta/|r_ij|) -> classical robust IRLS
    "encoder_only" : T = 0, prediction = x_A
"""
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

EPS = 1e-6


# -----------------------------------------------------------------------------
# helpers
# -----------------------------------------------------------------------------
def scatter_sum(src, index, n):
    out = torch.zeros((n,) + src.shape[1:], dtype=src.dtype, device=src.device)
    return out.index_add_(0, index, src)


def scatter_mean(src, index, n):
    s = scatter_sum(src, index, n)
    c = scatter_sum(torch.ones_like(src[:, :1]), index, n).clamp_min(1.0)
    return s / c


def mlp(i, h, o, layers=2):
    mods, d = [], i
    for _ in range(layers - 1):
        mods += [nn.Linear(d, h), nn.SiLU()]
        d = h
    mods.append(nn.Linear(d, o))
    return nn.Sequential(*mods)


# -----------------------------------------------------------------------------
# batching: many graphs -> one big disconnected graph
# -----------------------------------------------------------------------------
def collate(graphs, device="cpu"):
    xs, x0s, eis, eas, des, ys, am, batch, cen, sca, rots = [], [], [], [], [], [], [], [], [], [], []
    off = 0
    for b, g in enumerate(graphs):
        n = g["n"]
        R = g.get("rot", np.eye(2, dtype=np.float32))
        xs.append(g["x"]); x0s.append(g["x0"])
        eis.append(g["edge_index"] + off); eas.append(g["edge_attr"]); des.append(g["d_edge"])
        if "y_norm" in g:
            ys.append(g["y_norm"])
        else:
            ys.append(((g["true_xy"] - g["center"]) / g["scale"]) @ R)  # E(2)-canonicalised GT
        if "anchor_mask" in g:
            am.append(g["anchor_mask"])
        else:
            a = np.zeros(n, bool); a[g["anchors"]] = True; am.append(a)
        batch.append(np.full(n, b)); cen.append(g["center"]); sca.append(g["scale"]); rots.append(R)
        off += n
    T = lambda a, dt=torch.float32: torch.as_tensor(np.concatenate(a), dtype=dt, device=device)
    return dict(
        x=T(xs), x0=T(x0s), edge_index=torch.as_tensor(np.concatenate(eis, 1), device=device),
        edge_attr=T(eas), d_edge=T(des), y=T(ys), anchor_mask=T(am, torch.bool),
        batch=T(batch, torch.long), center=torch.as_tensor(np.stack(cen), device=device),
        scale=torch.as_tensor(np.array(sca), device=device),
        rot=torch.as_tensor(np.stack(rots), dtype=torch.float32, device=device),
        n_graphs=len(graphs), sizes=[g["n"] for g in graphs],
    )


# -----------------------------------------------------------------------------
# Encoder layer: edge-aware message passing (MPNN / GINE-style)
# -----------------------------------------------------------------------------
class MPLayer(nn.Module):
    """
    m_ij = MLP_m([h_i, h_j, e_ij])                 message along edge j -> i
    e_ij <- LN(e_ij + m_ij)                         edge state update
    h_i  <- LN(h_i + MLP_u([h_i, mean_j m_ij]))     node state update
    Unlike SAGEConv, the measured distance enters EVERY message via e_ij.
    """
    def __init__(self, H):
        super().__init__()
        self.msg = mlp(3 * H, H, H)
        self.upd = mlp(2 * H, H, H)
        self.ln_h, self.ln_e = nn.LayerNorm(H), nn.LayerNorm(H)

    def forward(self, h, e, ei):
        src, dst = ei
        m = self.msg(torch.cat([h[dst], h[src], e], -1))
        e = self.ln_e(e + m)
        agg = scatter_mean(m, dst, h.size(0))
        h = self.ln_h(h + self.upd(torch.cat([h, agg], -1)))
        return h, e


# -----------------------------------------------------------------------------
# Refiner step: one Guttman / dwMDS majorisation step with learned weights
# -----------------------------------------------------------------------------
class LearnedSMACOFStep(nn.Module):
    """
    For every node i (anchors are re-pinned afterwards):
        dhat_ij = ||x_i - x_j||,   r_ij = d_ij - dhat_ij
        t_ij    = x_j + d_ij * (x_i - x_j) / dhat_ij       (where edge ij wants x_i)
        w_ij    = softplus( MLP_w([h_i, h_j, e_ij, d_ij, dhat_ij, |r_ij|]) )
        x_i^+   = sum_j w_ij t_ij / sum_j w_ij             (Guttman transform, local)
        x_i     <- x_i + alpha (x_i^+ - x_i)               (damped Jacobi update)
    """
    def __init__(self, H, variant="learned", huber_delta=0.05):
        super().__init__()
        self.variant, self.delta = variant, huber_delta
        self.w_static = nn.Linear(3 * H, H)
        self.w_dyn = nn.Linear(3, H, bias=False)
        self.w_out = nn.Linear(H, 1)
        self.alpha_logit = nn.Parameter(torch.tensor(0.0))   # alpha = 0.5 at init

    def prepare(self, h, e, ei):
        if self.variant != "learned":
            return None
        src, dst = ei
        return self.w_static(torch.cat([h[dst], h[src], e], -1))

    def weights(self, static, d, dhat, r):
        if self.variant == "uniform":
            return torch.ones_like(d)
        if self.variant == "huber":
            return torch.clamp(self.delta / r.abs().clamp_min(EPS), max=1.0)
        z = F.silu(static + self.w_dyn(torch.stack([d, dhat, r.abs()], -1)))
        return F.softplus(self.w_out(z)).squeeze(-1) + 1e-4

    def forward(self, x, static, ei, d, anchor_mask, x_anchor):
        src, dst = ei
        diff = x[dst] - x[src]
        dhat = diff.norm(dim=-1).clamp_min(EPS)
        r = d - dhat
        w = self.weights(static, d, dhat, r)
        target = x[src] + d[:, None] * diff / dhat[:, None]
        num = scatter_sum(w[:, None] * target, dst, x.size(0))
        den = scatter_sum(w[:, None], dst, x.size(0)).clamp_min(EPS)
        alpha = torch.sigmoid(self.alpha_logit) if self.variant == "learned" else 0.5
        x_new = x + alpha * (num / den - x)
        return torch.where(anchor_mask[:, None], x_anchor, x_new), w


# -----------------------------------------------------------------------------
# Full model
# -----------------------------------------------------------------------------
class AnchorGNNLocalizer(nn.Module):
    def __init__(self, in_dim=19, edge_dim=3, H=128, enc_layers=6, refine_steps=10,
                 variant="learned", huber_delta=0.05):
        super().__init__()
        self.variant = variant
        self.T = 0 if variant == "encoder_only" else refine_steps
        self.node_in = mlp(in_dim, H, H)
        self.edge_in = mlp(edge_dim, H, H)
        self.layers = nn.ModuleList([MPLayer(H) for _ in range(enc_layers)])
        self.head = mlp(H, H, 2)
        # ONE shared step -> it is an iterative algorithm, so T can change at test time
        self.step = LearnedSMACOFStep(H, variant, huber_delta)

    def forward(self, B, T=None, return_all=False):
        x0, am, ei, d = B["x0"], B["anchor_mask"], B["edge_index"], B["d_edge"]
        x_anchor = x0                                   # anchors' x0 == true anchor xy
        h = self.node_in(B["x"])
        e = self.edge_in(B["edge_attr"])
        for L in self.layers:
            h, e = L(h, e, ei)
        xA = torch.where(am[:, None], x_anchor, x0 + self.head(h))   # stage-A estimate
        x, traj = xA, [xA]
        static = self.step.prepare(h, e, ei)
        for _ in range(self.T if T is None else T):
            x, _ = self.step(x, static, ei, d, am, x_anchor)
            traj.append(x)
        return (x, xA, traj) if return_all else (x, xA)


def to_original_units(B, x_norm):
    """E(2)-canonicalised prediction -> original coordinates, list of (n_i, 2) arrays."""
    out, off = [], 0
    for b, n in enumerate(B["sizes"]):
        R = B["rot"][b]
        xb = (x_norm[off:off + n] @ R.T) * B["scale"][b] + B["center"][b]
        out.append(xb.detach().cpu().numpy()); off += n
    return out


# -----------------------------------------------------------------------------
# Losses
# -----------------------------------------------------------------------------
def supervised_loss(x, B, delta):
    """Huber on coordinates of NON-anchor nodes (normalised units)."""
    m = ~B["anchor_mask"]
    return F.huber_loss(x[m], B["y"][m], delta=delta)


def selfsup_stress_loss(x, B, delta):
    """Asymmetric NLOS-aware Huber stress loss on observed edges:
    Since NLOS biases are positive (d_edge >= d_true), dhat > d_edge is penalized
    quadratically, while dhat < d_edge uses robust Huber to discount positive NLOS outliers."""
    src, dst = B["edge_index"]
    dhat = (x[dst] - x[src]).norm(dim=-1).clamp_min(EPS)
    res = dhat - B["d_edge"]
    quad_mask = res >= 0.0
    loss_pos = 0.5 * (res ** 2)
    loss_neg = F.huber_loss(dhat, B["d_edge"], delta=delta, reduction="none")
    return torch.where(quad_mask, loss_pos, loss_neg).mean()


In [ ]:
%%writefile baselines.py
"""
baselines.py -- existing methods, all run on EXACTLY the same graphs.

Every function takes one graph dict g (from common.make_graph) and returns
an (n, 2) array of coordinates in ORIGINAL units. They use only
g["edges"], g["d_obs"], g["anchors"], g["anchor_xy"] -- never g["true_xy"].

    mds_map            Shang et al. 2003: geodesic completion + classical MDS + Procrustes
    mds_map_stress     MDS-MAP followed by full stress minimisation (L-BFGS)   <- strong
    mds_map_huber      same, robust Huber stress (classical NLOS-robust choice)
    dv_dwmds           DV-distance multilateration + 300 dwMDS iterations (w = 1)
                       = the classical algorithm our refiner unrolls, with NO learning
    sdp_biswas_ye      Biswas & Ye 2004 SDP relaxation (L1 slack), cvxpy
    yan_gcn            Yan et al. ICASSP 2021 GCN (per-graph, anchors are the labels)
    pergraph_sage      your previous approach: GraphSAGE trained per graph on stress
"""
import time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.optimize import minimize
from common import geodesic_matrix, multilaterate


# -----------------------------------------------------------------------------
# helpers
# -----------------------------------------------------------------------------
def classical_mds(D, d=2):
    n = len(D)
    J = np.eye(n) - 1.0 / n
    B = -0.5 * J @ (D ** 2) @ J
    w, V = np.linalg.eigh((B + B.T) / 2)
    o = np.argsort(w)[::-1][:d]
    return V[:, o] * np.sqrt(np.clip(w[o], 0, None))


def procrustes_O2(X, anchors, A):
    """Best rotation OR reflection + translation mapping X[anchors] onto A."""
    S = X[anchors]
    sc, ac = S.mean(0), A.mean(0)
    U, _, Vt = np.linalg.svd((S - sc).T @ (A - ac))
    best, br = None, np.inf
    for sg in (1.0, -1.0):
        V = Vt.copy(); V[-1] *= sg
        R = V.T @ U.T
        r = np.linalg.norm((S - sc) @ R.T - (A - ac))
        if r < br:
            best, br = R, r
    return (X - sc) @ best.T + ac


def geodesic_matrix_torch(n, E, d, device="cpu"):
    dev = torch.device(device if (device != "cpu" and torch.cuda.is_available()) else "cpu")
    D = torch.full((n, n), float("inf"), dtype=torch.float32, device=dev)
    D.fill_diagonal_(0.0)
    src = torch.as_tensor(E[:, 0], dtype=torch.long, device=dev)
    dst = torch.as_tensor(E[:, 1], dtype=torch.long, device=dev)
    dd = torch.as_tensor(d, dtype=torch.float32, device=dev)
    D[src, dst] = dd
    D[dst, src] = dd
    for k in range(n):
        D = torch.minimum(D, D[:, k:k+1] + D[k:k+1, :])
    return D


def classical_mds_torch(D, d=2, device="cpu"):
    n = D.shape[0]
    dev = torch.device(device if (device != "cpu" and torch.cuda.is_available()) else "cpu")
    Dt = torch.as_tensor(D, dtype=torch.float32, device=dev)
    J = torch.eye(n, device=dev) - (1.0 / n)
    B = -0.5 * J @ (Dt ** 2) @ J
    sym_B = (B + B.T) / 2
    try:
        w, V = torch.linalg.eigh(sym_B)
    except Exception:
        # cuSOLVER lazy wrapper race condition or CUDA exception -> fallback to numpy
        w_np, V_np = np.linalg.eigh(sym_B.detach().cpu().numpy())
        w = torch.as_tensor(w_np, device=dev)
        V = torch.as_tensor(V_np, device=dev)
    idx = torch.arange(n - 1, n - d - 1, -1, device=dev)
    val = torch.clamp(w[idx], min=0.0)
    X = V[:, idx] * torch.sqrt(val)
    return X


def procrustes_torch(X, anchors, A, device="cpu"):
    dev = X.device
    anc = torch.as_tensor(anchors, dtype=torch.long, device=dev)
    At = torch.as_tensor(A, dtype=torch.float32, device=dev)
    S = X[anc]
    sc = S.mean(0)
    ac = At.mean(0)
    H = (S - sc).T @ (At - ac)
    try:
        U, _, Vt = torch.linalg.svd(H)
    except Exception:
        U_np, _, Vt_np = np.linalg.svd(H.detach().cpu().numpy())
        U = torch.as_tensor(U_np, device=dev)
        Vt = torch.as_tensor(Vt_np, device=dev)
    best_R = None
    best_err = float("inf")
    for sg in (1.0, -1.0):
        V_curr = Vt.clone()
        V_curr[-1, :] *= sg
        R = V_curr.T @ U.T
        err = torch.norm((S - sc) @ R.T - (At - ac))
        if err < best_err:
            best_R = R
            best_err = err
    X_aligned = (X - sc) @ best_R.T + ac
    X_aligned[anc] = At
    return X_aligned


def _pin(g, X):
    X = X.copy(); X[g["anchors"]] = g["anchor_xy"]; return X


def stress_lbfgs(g, X0, robust=False, max_iter=3000, device="cpu"):
    """Minimise sum_edges rho(||x_i - x_j|| - d_ij) over NON-anchor nodes.
    Uses compiled C/Fortran L-BFGS-B (10x faster than GPU for small 94-variable graphs)."""
    n, E, d = g["n"], g["edges"], g["d_obs"].astype(float)
    I, J = E[:, 0], E[:, 1]
    free = np.ones(n, bool); free[g["anchors"]] = False
    Xfix = _pin(g, X0).astype(float)
    delta = 0.1 * np.median(d)

    def f(z):
        X = Xfix.copy(); X[free] = z.reshape(-1, 2)
        diff = X[I] - X[J]
        dh = np.sqrt((diff ** 2).sum(1) + 1e-12)
        r = dh - d
        if robust:
            a = np.abs(r)
            val = np.where(a <= delta, 0.5 * r ** 2, delta * (a - 0.5 * delta)).sum()
            dr = np.where(a <= delta, r, delta * np.sign(r))
        else:
            val, dr = (r ** 2).sum(), 2 * r
        c = (dr / dh)[:, None] * diff
        G = np.zeros_like(X)
        np.add.at(G, I, c); np.add.at(G, J, -c)
        return val, G[free].ravel()

    z = minimize(f, Xfix[free].ravel(), jac=True, method="L-BFGS-B",
                 options={"maxiter": max_iter}).x
    X = Xfix.copy(); X[free] = z.reshape(-1, 2)
    return X


# -----------------------------------------------------------------------------
# baselines
# -----------------------------------------------------------------------------
def mds_map(g, device="cpu"):
    D = geodesic_matrix(g["n"], g["edges"], g["d_obs"].astype(float))
    X = classical_mds(D)
    return _pin(g, procrustes_O2(X, g["anchors"], g["anchor_xy"]))


def mds_map_stress(g, device="cpu"):
    return stress_lbfgs(g, mds_map(g), robust=False)


def mds_map_huber(g, device="cpu"):
    return stress_lbfgs(g, mds_map(g), robust=True)




def dv_dwmds(g, iters=300, alpha=0.5, device="cpu"):
    n, E, d = g["n"], g["edges"], g["d_obs"].astype(float)
    G = geodesic_matrix(n, E, d)[:, g["anchors"]]
    X = _pin(g, multilaterate(g["anchor_xy"].astype(float), G))
    if device != "cpu" and torch.cuda.is_available():
        dev = torch.device(device)
        src = torch.as_tensor(np.r_[E[:, 0], E[:, 1]], dtype=torch.long, device=dev)
        dst = torch.as_tensor(np.r_[E[:, 1], E[:, 0]], dtype=torch.long, device=dev)
        dd = torch.as_tensor(np.r_[d, d], dtype=torch.float32, device=dev)
        deg = torch.bincount(dst, minlength=n).float().unsqueeze(1).to(dev)
        Xt = torch.as_tensor(X, dtype=torch.float32, device=dev)
        anc = torch.as_tensor(g["anchors"], dtype=torch.long, device=dev)
        axy = torch.as_tensor(g["anchor_xy"], dtype=torch.float32, device=dev)
        for _ in range(iters):
            diff = Xt[dst] - Xt[src]
            dh = diff.norm(dim=-1, keepdim=True).clamp_min(1e-9)
            t = Xt[src] + dd.unsqueeze(1) * diff / dh
            num = torch.zeros_like(Xt)
            num.index_add_(0, dst, t)
            Xt = Xt + alpha * (num / deg - Xt)
            Xt[anc] = axy
        return Xt.cpu().numpy()
    src, dst = np.r_[E[:, 0], E[:, 1]], np.r_[E[:, 1], E[:, 0]]
    dd = np.r_[d, d]
    deg = np.bincount(dst, minlength=n)[:, None]
    for _ in range(iters):
        diff = X[dst] - X[src]
        dh = np.linalg.norm(diff, axis=1).clip(1e-9)
        t = X[src] + dd[:, None] * diff / dh[:, None]
        num = np.zeros_like(X); np.add.at(num, dst, t)
        X = _pin(g, X + alpha * (num / deg - X))
    return X


def sdp_biswas_ye(g, device="cpu"):
    """
    Z = [[I_2, X], [X^T, Y]] >= 0,  X = unknown positions (2 x m).
      unknown-unknown edge: Y_ii + Y_jj - 2 Y_ij      = d^2 + s_ij
      anchor-unknown edge : ||a||^2 - 2 a^T x_i + Y_ii = d^2 + s_ij
    minimise sum |s_ij|   (L1 slack = robust to outliers)
    """
    import cvxpy as cp
    n, E, d = g["n"], g["edges"], g["d_obs"].astype(float)
    anchors = list(g["anchors"])
    is_a = np.zeros(n, bool); is_a[anchors] = True
    axy = np.zeros((n, 2)); axy[anchors] = g["anchor_xy"]
    unk = np.nonzero(~is_a)[0]
    pos = -np.ones(n, int); pos[unk] = np.arange(len(unk)) + 2   # index inside Z
    m = len(unk)
    Z = cp.Variable((m + 2, m + 2), PSD=True)
    dg = cp.diag(Z)
    uu = ~is_a[E[:, 0]] & ~is_a[E[:, 1]]                 # unknown-unknown edges
    au = is_a[E[:, 0]] ^ is_a[E[:, 1]]                    # anchor-unknown edges
    res = []
    if uu.any():
        P, Q = pos[E[uu, 0]], pos[E[uu, 1]]
        res.append(dg[P] + dg[Q] - 2 * Z[P, Q] - d[uu] ** 2)
    if au.any():
        Ea = E[au]
        a_node = np.where(is_a[Ea[:, 0]], Ea[:, 0], Ea[:, 1])
        u_node = np.where(is_a[Ea[:, 0]], Ea[:, 1], Ea[:, 0])
        K = pos[u_node]; a = axy[a_node]
        res.append((a ** 2).sum(1) - 2 * (cp.multiply(a[:, 0], Z[0, K]) + cp.multiply(a[:, 1], Z[1, K]))
                   + dg[K] - d[au] ** 2)
    cons = [Z[0, 0] == 1, Z[1, 1] == 1, Z[0, 1] == 0]
    prob = cp.Problem(cp.Minimize(sum(cp.sum(cp.abs(r)) for r in res)), cons)
    try:
        use_gpu = (device != "cpu" and torch.cuda.is_available())
        try:
            if use_gpu:
                prob.solve(solver=cp.SCS, max_iters=1000, eps=1e-3, verbose=False, gpu=True)
            else:
                prob.solve(solver=cp.SCS, max_iters=1000, eps=1e-3, verbose=False)
        except Exception:
            prob.solve(solver=cp.SCS, max_iters=1000, eps=1e-3, verbose=False)
        Xu = np.asarray(Z.value[:2, 2:]).T
    except Exception:
        return mds_map(g, device=device)
    X = np.zeros((n, 2)); X[unk] = Xu
    return _pin(g, X)



def yan_gcn(g, epochs=200, hidden=256, lr=0.01, dropout=0.5, seed=0, device="cpu"):
    """
    Yan et al. 2021 (ICASSP), reproduced on our graphs:
      features = rows of the (thresholded = observed) distance matrix, row-normalised
      2-layer GCN with augmented normalised adjacency, trained per graph,
      loss = MSE on the ANCHOR nodes only (here: 3 labels).
    Paper used 2000 hidden units; 256 is used here for speed.
    """
    torch.manual_seed(seed)
    dev = torch.device(device if (device != "cpu" and torch.cuda.is_available()) else "cpu")
    n, E, d = g["n"], g["edges"], g["d_obs"].astype(np.float32)
    Xd = np.zeros((n, n), np.float32)
    Xd[E[:, 0], E[:, 1]] = d; Xd[E[:, 1], E[:, 0]] = d
    Xf_np = Xd / np.maximum(np.linalg.norm(Xd, axis=1, keepdims=True), 1e-9)
    A = (Xd > 0).astype(np.float32) + np.eye(n, dtype=np.float32)
    Dm = 1.0 / np.sqrt(A.sum(1))
    Ah = torch.tensor(Dm[:, None] * A * Dm[None, :], device=dev)
    Xf = torch.tensor(Xf_np, device=dev)
    W1, W2 = nn.Linear(n, hidden).to(dev), nn.Linear(hidden, 2).to(dev)
    opt = torch.optim.Adam(list(W1.parameters()) + list(W2.parameters()), lr=lr)
    a = torch.tensor(g["anchors"], dtype=torch.long, device=dev)
    ya = torch.tensor(g["anchor_xy"], dtype=torch.float32, device=dev)
    def fwd(train):                       # R = A_h relu(A_h X W1) W2   (paper eq. 5)
        h = F.dropout(F.relu(W1(Ah @ Xf)), dropout, train)
        return Ah @ W2(h)
    for _ in range(epochs):
        opt.zero_grad(); loss = F.mse_loss(fwd(True)[a], ya); loss.backward(); opt.step()
    with torch.no_grad():
        return _pin(g, fwd(False).cpu().numpy())


class _MPNNLocNet(nn.Module):
    """
    MPNN-Loc Architecture from:
    Xu & Lin (IFAC 2025): 'Graph Neural Network based Localization for Layout-Agnostic Sensor Networks'
    """
    def __init__(self, n, in_dim=2, hidden=200, dropout=0.5):
        super().__init__()
        self.n = n
        self.dropout = dropout
        
        # Layer 1: t=1, M^1 dimension = 2 * in_dim + 1 = 5
        m1_dim = 2 * in_dim + 1
        self.W_node1 = nn.Linear(m1_dim, hidden)
        self.W_edge1 = nn.Linear(m1_dim, n)
        
        # Layer 2: Readout at t=Tl=2, M^2 dimension = 2 * hidden + 1 = 401
        m2_dim = 2 * hidden + 1
        self.W_readout = nn.Linear(m2_dim, 2)

    def forward(self, H0, E0, A_hat):
        # A_hat: row-normalized adjacency matrix (N, N)
        # H0: initial node features (N, 2)
        # E0: initial edge distance matrix (N, N)
        
        # --- Layer 1 ---
        AE0_diag = torch.diagonal(A_hat @ E0).unsqueeze(-1)
        M1 = torch.cat([H0, A_hat @ H0, AE0_diag], dim=-1)   # (N, 5)
        
        # Node update (Eq. 11)
        H1 = F.relu(self.W_node1(M1))
        H1 = F.dropout(H1, p=self.dropout, training=self.training)
        
        # Edge update (Eq. 12)
        E1_raw = self.W_edge1(M1)                            # (N, N)
        E1 = F.relu(A_hat * E1_raw)                          # Hadamard product A_hat \odot (...)
        
        # --- Layer 2 / Readout (Eq. 10 & 13) ---
        AE1_diag = torch.diagonal(A_hat @ E1).unsqueeze(-1)
        M2 = torch.cat([H1, A_hat @ H1, AE1_diag], dim=-1)   # (N, 2*hidden + 1)
        
        P_hat = self.W_readout(M2)                           # (N, 2)
        return P_hat


def _correlation_refinement(P_hat, P_L, A, anchors, agents, dev):
    """
    Correlation-based refinement module (Eq. 14-17 in Xu & Lin, IFAC 2025).
    P'_U = P_hat_U - Gamma_UU^{-1} Gamma_UL (P_L - P_hat_L)
    where Gamma = I - K^{-1/2} A K^{-1/2} is the symmetric normalized Laplacian.
    """
    n = A.shape[0]
    deg = A.sum(dim=-1)
    deg_inv_sqrt = torch.where(deg > 0, 1.0 / torch.sqrt(deg), torch.zeros_like(deg))
    A_sym = deg_inv_sqrt.unsqueeze(-1) * A * deg_inv_sqrt.unsqueeze(0)
    Gamma = torch.eye(n, device=dev) - A_sym
    
    # Submatrices
    Gamma_UU = Gamma[agents][:, agents]
    Gamma_UL = Gamma[agents][:, anchors]
    
    # Anchors difference: (P_L - P_hat_L)
    diff_L = P_L - P_hat[anchors]
    rhs = Gamma_UL @ diff_L                                # (N_agents, 2)
    
    # Regularized solve for numerical stability
    reg = 1e-4 * torch.eye(len(agents), device=dev)
    try:
        delta_U = torch.linalg.solve(Gamma_UU + reg, rhs)
    except Exception:
        delta_U = torch.linalg.lstsq(Gamma_UU + reg, rhs).solution
        
    P_refined = P_hat.clone()
    P_refined[agents] = P_hat[agents] - delta_U
    P_refined[anchors] = P_L
    return P_refined


def mpnn_loc(g, epochs=200, hidden=200, lr=0.001, dropout=0.5, refine=True, seed=0, device="cpu"):
    """
    MPNN-Loc algorithm from Xu & Lin (IFAC 2025):
    'Graph Neural Network based Localization for Layout-Agnostic Sensor Networks'
    
    Consists of:
    1. Node feature initialization (anchor coords, nearest-anchor coords, or area center)
    2. 2-layer MPNN with joint node & edge updates
    3. Semi-supervised training minimizing MSE on anchor predictions
    4. Nonparametric correlation-based refinement using symmetric normalized Laplacian
    """
    torch.manual_seed(seed)
    dev = torch.device(device if (device != "cpu" and torch.cuda.is_available()) else "cpu")
    
    n = g["n"]
    E = g["edges"]
    d = g["d_obs"].astype(np.float32)
    anchors_list = list(g["anchors"])
    anchors = torch.tensor(anchors_list, dtype=torch.long, device=dev)
    agents_list = [i for i in range(n) if i not in anchors_list]
    agents = torch.tensor(agents_list, dtype=torch.long, device=dev)
    P_L = torch.tensor(g["anchor_xy"], dtype=torch.float32, device=dev)
    
    # 1. Adjacency and Distance Matrices
    A_np = np.zeros((n, n), dtype=np.float32)
    D_np = np.zeros((n, n), dtype=np.float32)
    A_np[E[:, 0], E[:, 1]] = 1.0
    A_np[E[:, 1], E[:, 0]] = 1.0
    D_np[E[:, 0], E[:, 1]] = d
    D_np[E[:, 1], E[:, 0]] = d
    
    # Row-normalized adjacency A_hat = K^{-1} A (Eq. 3, page 3)
    k_np = np.maximum(A_np.sum(axis=1, keepdims=True), 1e-6)
    A_hat_np = A_np / k_np
    
    A = torch.tensor(A_np, device=dev)
    A_hat = torch.tensor(A_hat_np, device=dev)
    E0 = torch.tensor(D_np, device=dev)
    
    # 2. Node feature initialization X (Eq. 5-9)
    # Anchor: X_i = P_i
    # Agent: X_i = P_vina if connected to any anchor, else P_center
    P_center = g["anchor_xy"].mean(axis=0)
    X_init = np.tile(P_center, (n, 1)).astype(np.float32)
    for a_idx, a_node in enumerate(anchors_list):
        X_init[a_node] = g["anchor_xy"][a_idx]
        
    for ag in agents_list:
        anc_nbrs = [nbr for nbr in anchors_list if A_np[ag, nbr] > 0]
        if anc_nbrs:
            vina = min(anc_nbrs, key=lambda a_nbr: D_np[ag, a_nbr])
            vina_idx = anchors_list.index(vina)
            X_init[ag] = g["anchor_xy"][vina_idx]
            
    H0 = torch.tensor(X_init, device=dev)
    
    # 3. Model & Optimizer
    model = _MPNNLocNet(n, in_dim=2, hidden=hidden, dropout=dropout).to(dev)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    # 4. Training (Eq. 18: loss = MSE(P_hat_L, P_L))
    model.train()
    for _ in range(epochs):
        optimizer.zero_grad()
        P_hat = model(H0, E0, A_hat)
        loss = F.mse_loss(P_hat[anchors], P_L)
        loss.backward()
        optimizer.step()
        
    # 5. Inference & Correlation-based refinement (Eq. 14-17)
    model.eval()
    with torch.no_grad():
        P_hat = model(H0, E0, A_hat)
        if refine:
            P_out = _correlation_refinement(P_hat, P_L, A, anchors, agents, dev)
        else:
            P_out = P_hat
            P_out[anchors] = P_L
        
    return P_out.cpu().numpy()


class _SAGE(nn.Module):
    def __init__(self, H=64):
        super().__init__()
        self.l1s, self.l1n = nn.Linear(2, H), nn.Linear(2, H, bias=False)
        self.l2s, self.l2n = nn.Linear(H, H), nn.Linear(H, H, bias=False)
        self.out = nn.Linear(H, 2)
    def forward(self, x, An):
        h = F.relu(self.l1s(x) + self.l1n(An @ x))
        h = F.relu(self.l2s(h) + self.l2n(An @ h))
        return x + self.out(h)


def pergraph_sage(g, epochs=1000, lr=1e-3, seed=0, device="cpu"):
    """Your previous pipeline (step1_corrected): MDS-MAP init -> GraphSAGE with a
    residual output, trained from scratch on THIS graph with Huber stress + anchors."""
    torch.manual_seed(seed)
    dev = torch.device(device if (device != "cpu" and torch.cuda.is_available()) else "cpu")
    n, E = g["n"], g["edges"]
    x0 = torch.tensor(mds_map(g), dtype=torch.float32, device=dev)
    A = np.zeros((n, n), np.float32); A[E[:, 0], E[:, 1]] = 1; A[E[:, 1], E[:, 0]] = 1
    An = torch.tensor(A / A.sum(1, keepdims=True), device=dev)
    I = torch.tensor(E[:, 0], dtype=torch.long, device=dev)
    J = torch.tensor(E[:, 1], dtype=torch.long, device=dev)
    d = torch.tensor(g["d_obs"], dtype=torch.float32, device=dev); s = d.mean()
    a = torch.tensor(g["anchors"], dtype=torch.long, device=dev)
    ya = torch.tensor(g["anchor_xy"], dtype=torch.float32, device=dev)
    model = _SAGE().to(dev); opt = torch.optim.Adam(model.parameters(), lr=lr)
    for _ in range(epochs):
        opt.zero_grad()
        X = model(x0, An)
        dh = (X[I] - X[J]).norm(dim=-1).clamp_min(1e-6)
        loss = F.huber_loss((dh - d) / s, torch.zeros_like(d)) + (((X[a] - ya) / s) ** 2).sum(-1).mean()
        loss.backward(); opt.step()
    with torch.no_grad():
        return _pin(g, model(x0, An).cpu().numpy())


def naive_classical_mds(g, device="cpu"):
    """Classical MDS (Algorithm 1) directly on observed distances (unobserved = 0)."""
    dev = torch.device(device if (device != "cpu" and torch.cuda.is_available()) else "cpu")
    n, E, d = g["n"], g["edges"], g["d_obs"].astype(np.float32)
    src = torch.as_tensor(E[:, 0], dtype=torch.long, device=dev)
    dst = torch.as_tensor(E[:, 1], dtype=torch.long, device=dev)
    Dt = torch.zeros((n, n), dtype=torch.float32, device=dev)
    dd = torch.as_tensor(d, dtype=torch.float32, device=dev)
    Dt[src, dst] = dd
    Dt[dst, src] = dd
    X = classical_mds_torch(Dt, d=2, device=dev)
    return procrustes_torch(X, g["anchors"], g["anchor_xy"], device=dev).cpu().numpy()


def rank_complete_edm(g, max_iters=100, tol=1e-5, device="cpu"):
    """Dokmanic et al. Algorithm 2: Alternating Rank-Based EDM Completion."""
    dev = torch.device(device if (device != "cpu" and torch.cuda.is_available()) else "cpu")
    n, E, d = g["n"], g["edges"], g["d_obs"].astype(np.float32)
    W = torch.zeros((n, n), dtype=torch.bool, device=dev)
    src = torch.as_tensor(E[:, 0], dtype=torch.long, device=dev)
    dst = torch.as_tensor(E[:, 1], dtype=torch.long, device=dev)
    W[src, dst] = True
    W[dst, src] = True
    d2 = torch.as_tensor(d ** 2, dtype=torch.float32, device=dev)
    D2 = torch.zeros((n, n), dtype=torch.float32, device=dev)
    D2[src, dst] = d2
    D2[dst, src] = d2
    mu = d2.mean()
    D_cur = torch.where(W, D2, mu)
    D_cur.fill_diagonal_(0.0)
    r = 4  # rank d+2
    for _ in range(max_iters):
        sym_D = (D_cur + D_cur.T) / 2
        try:
            w, V = torch.linalg.eigh(sym_D)
        except Exception:
            w_np, V_np = np.linalg.eigh(sym_D.detach().cpu().numpy())
            w = torch.as_tensor(w_np, device=dev)
            V = torch.as_tensor(V_np, device=dev)
        idx = torch.topk(w.abs(), r).indices
        D_next = (V[:, idx] * w[idx]) @ V[:, idx].T
        D_next[W] = D2[W]
        D_next.fill_diagonal_(0.0)
        D_next.clamp_min_(0.0)
        if torch.norm(D_next - D_cur) / (torch.norm(D_cur) + 1e-9) < tol:
            break
        D_cur = D_next
    D_est = torch.sqrt(D_cur)
    X = classical_mds_torch(D_est, d=2, device=dev)
    return procrustes_torch(X, g["anchors"], g["anchor_xy"], device=dev).cpu().numpy()


def optspace_edm(g, rank=4, device="cpu"):
    """Dokmanic et al. Algorithm 3: OptSpace EDM Completion."""
    dev = torch.device(device if (device != "cpu" and torch.cuda.is_available()) else "cpu")
    n, E, d = g["n"], g["edges"], g["d_obs"].astype(np.float32)
    W = torch.zeros((n, n), dtype=torch.bool, device=dev)
    src = torch.as_tensor(E[:, 0], dtype=torch.long, device=dev)
    dst = torch.as_tensor(E[:, 1], dtype=torch.long, device=dev)
    W[src, dst] = True
    W[dst, src] = True
    d2 = torch.as_tensor(d ** 2, dtype=torch.float32, device=dev)
    D2 = torch.zeros((n, n), dtype=torch.float32, device=dev)
    D2[src, dst] = d2
    D2[dst, src] = d2
    total_pairs = n * (n - 1) / 2.0
    p = max(len(E) / total_pairs, 1e-4)
    M_scaled = D2 / p
    deg = W.sum(dim=1).float()
    avg_deg = deg.mean()
    trim_mask = deg > (2.0 * avg_deg)
    M_trimmed = M_scaled.clone()
    if trim_mask.any():
        M_trimmed[trim_mask, :] = 0.0
        M_trimmed[:, trim_mask] = 0.0
    try:
        U, s, Vt = torch.linalg.svd(M_trimmed, full_matrices=False)
    except Exception:
        U_np, s_np, Vt_np = np.linalg.svd(M_trimmed.detach().cpu().numpy(), full_matrices=False)
        U = torch.as_tensor(U_np, device=dev)
        s = torch.as_tensor(s_np, device=dev)
        Vt = torch.as_tensor(Vt_np, device=dev)
    U_r, s_r, Vt_r = U[:, :rank], s[:rank], Vt[:rank, :]
    D_est = U_r @ torch.diag(s_r) @ Vt_r
    D_est[W] = D2[W]
    D_est.fill_diagonal_(0.0)
    D_est.clamp_min_(0.0)
    D_dist = torch.sqrt(D_est)
    X = classical_mds_torch(D_dist, d=2, device=dev)
    return procrustes_torch(X, g["anchors"], g["anchor_xy"], device=dev).cpu().numpy()


def alternating_descent(g, max_iters=50, seed=0, device="cpu"):
    """Dokmanic et al. Algorithm 4: Alternating Descent on s-stress.
    Fixed initialization: uses small Gaussian perturbation around anchor centroid
    instead of all-zeros to avoid the zero-gradient saddle point at iteration 0."""
    dev = torch.device(device if (device != "cpu" and torch.cuda.is_available()) else "cpu")
    torch.manual_seed(seed)
    n, E, d = g["n"], g["edges"], g["d_obs"].astype(np.float32)
    src = torch.as_tensor(E[:, 0], dtype=torch.long, device=dev)
    dst = torch.as_tensor(E[:, 1], dtype=torch.long, device=dev)
    d2 = torch.as_tensor(d ** 2, dtype=torch.float32, device=dev)
    anc = torch.as_tensor(g["anchors"], dtype=torch.long, device=dev)
    axy = torch.as_tensor(g["anchor_xy"], dtype=torch.float32, device=dev)
    is_free = torch.ones(n, dtype=torch.bool, device=dev)
    is_free[anc] = False
    free_idx = torch.nonzero(is_free).squeeze(-1)
    init_xy = axy.mean(0, keepdim=True) + 0.1 * torch.randn((len(free_idx), 2), dtype=torch.float32, device=dev)
    X_free = torch.nn.Parameter(init_xy)
    optimizer = torch.optim.LBFGS([X_free], max_iter=max_iters, lr=1.0, line_search_fn="strong_wolfe")
    def closure():
        optimizer.zero_grad()
        X_full = torch.zeros((n, 2), dtype=torch.float32, device=dev)
        X_full[anc] = axy
        X_full[free_idx] = X_free
        diff = X_full[src] - X_full[dst]
        dhat2 = (diff ** 2).sum(dim=-1)
        loss = ((dhat2 - d2) ** 2).sum()
        loss.backward()
        return loss
    try:
        optimizer.step(closure)
    except Exception:
        pass
    X_final = torch.zeros((n, 2), dtype=torch.float32, device=dev)
    X_final[anc] = axy
    X_final[free_idx] = X_free.detach()
    return X_final.cpu().numpy()


def get_baselines(cfg, device="cpu"):
    B = {
        "MDS-MAP": lambda g: mds_map(g, device=device),
        "MDS-MAP + stress (L-BFGS)": lambda g: mds_map_stress(g, device=device),
        "MDS-MAP + Huber stress": lambda g: mds_map_huber(g, device=device),
        "DV-distance + dwMDS (no learning)": lambda g: dv_dwmds(g, device=device),
        "Classical MDS (direct)": lambda g: naive_classical_mds(g, device=device),
        "Rank Alternation (EDM book Alg 2)": lambda g: rank_complete_edm(g, device=device),
        "OptSpace (EDM book Alg 3)": lambda g: optspace_edm(g, device=device),
        "Alternating Descent (EDM book Alg 4)": lambda g: alternating_descent(g, device=device),
        "Yan et al. GCN (3 anchor labels)": lambda g: yan_gcn(g, epochs=cfg.YAN_GCN_EPOCHS, device=device),
    }
    if getattr(cfg, "RUN_MPNN_LOC", True):
        B["MPNN-Loc (Xu & Lin 2025)"] = lambda g: mpnn_loc(
            g, epochs=getattr(cfg, "MPNN_LOC_EPOCHS", 200), device=device
        )
    if cfg.RUN_SDP:
        B["SDP (Biswas-Ye)"] = lambda g: sdp_biswas_ye(g, device=device)
    if cfg.RUN_PERGRAPH_SAGE:
        B["Per-graph GraphSAGE (previous)"] = lambda g: pergraph_sage(g, epochs=cfg.PERGRAPH_EPOCHS, device=device)
    return B


def timed(fn, g):
    t = time.perf_counter(); X = fn(g); return X, time.perf_counter() - t


In [ ]:
%%writefile step1_data.py
"""
STEP 1 -- build the datasets.

    train / val : many SMALL graphs, mixed everything (n in 30..80, missing in
                  10..90 %, noise model, graph type) -> the GNN learns the
                  general skill, not one graph.
    test        : TEST_PER_LEVEL graphs for every (scenario, missing level).
                  EVERY method (ours and all baselines) is scored on these
                  exact same graphs in step 3.
    generalise  : bigger graphs (n = 100, 200) never seen in training.
    ba          : Barabasi-Albert graphs, m = 1..5 (rigidity threshold).
    pools       : (RUN_TRANSFER) one train + val pool per graph type, for the
                  train-on-some-types / test-on-all transfer experiment.

Each graph stores what methods may use (edges, d_obs, anchors, anchor_xy,
features) and, separately, true_xy (used ONLY for the training loss on
synthetic training graphs and for scoring).

Run:  python step1_data.py
"""
import pickle, time, os
import numpy as np
import config as C
from common import make_graph, ba_reachable


def save(obj, name):
    path = os.path.join(C.DATA_DIR, name)
    with open(path, "wb") as f:
        pickle.dump(obj, f, protocol=4)
    print(f"  saved {len(obj):5d} graphs -> {path}")


def mixed_set(rng, count, types=None, label=""):
    types = types or C.GRAPH_TYPES
    out = []
    noise_names = list(C.NOISE_MODELS)
    log_step = max(1, count // 5)
    t0 = time.time()
    while len(out) < count:
        n = int(rng.integers(C.TRAIN_N_RANGE[0], C.TRAIN_N_RANGE[1] + 1))
        gt = types[rng.integers(len(types))]
        lo, hi = C.TRAIN_MISSING
        if gt == "ba":                     # BA cannot go below ~50 % missing (see common.ba_m_for)
            lo = max(lo, 0.5)
        mp = float(rng.uniform(lo, hi))
        nz = noise_names[rng.integers(len(noise_names))]
        g = make_graph(rng, n, gt, mp, C.NOISE_MODELS[nz], C.N_ANCHORS, C.COORD_RANGE)
        g["noise"] = nz
        out.append(g)
        if len(out) % log_step == 0 or len(out) == count:
            print(f"    ... {label} generated {len(out)}/{count} graphs ({len(out)/count*100:.0f}%) in {time.time() - t0:.1f}s", flush=True)
    return out


def scenario_set(rng, n, scenarios, levels, per_level):
    out = []
    for gt, nz in scenarios:
        for mp in levels:
            if gt == "ba" and not ba_reachable(n, mp):
                print(f"  skipping ba/{nz} at {mp:.0%} missing: BA needs >= ~50 % missing at n = {n}", flush=True)
                continue
            for k in range(per_level):
                g = make_graph(rng, n, gt, mp, C.NOISE_MODELS[nz], C.N_ANCHORS, C.COORD_RANGE)
                g.update(noise=nz, scenario=f"{gt}/{nz}", graph_id=k)
                out.append(g)
            print(f"    test scenario {gt:6s} | noise: {nz:5s} | missing: {mp:.0%} ({len(out)} test graphs generated)", flush=True)
    return out


def summary(graphs, title):
    import pandas as pd
    df = pd.DataFrame([dict(scenario=g.get("scenario", "mixed"), missing=g["missing_nominal"],
                            edges=len(g["edges"]), rigid=g["globally_rigid"],
                            certified=g["certified"].mean()) for g in graphs])
    print(f"\n  {title}: how many graphs are even uniquely solvable?", flush=True)
    df["missing_actual"] = [g["missing_actual"] for g in graphs]
    print(df.groupby(["scenario", "missing"]).agg(edges=("edges", "mean"),
          missing_actual=("missing_actual", "mean"),
          globally_rigid=("rigid", "mean"), certified_nodes=("certified", "mean"))
          .round(2).to_string(), flush=True)


if __name__ == "__main__":
    t0 = time.time()
    print("STEP 1: generating datasets", flush=True)
    print("  [1/4] Generating training set...", flush=True)
    save(mixed_set(np.random.default_rng(C.SEED), C.N_TRAIN, label="train"), "train.pkl")
    print("  [2/4] Generating validation set...", flush=True)
    save(mixed_set(np.random.default_rng(C.SEED + 1), C.N_VAL, label="val"), "val.pkl")

    if C.RUN_TRANSFER:
        print("  Generating transfer pools...", flush=True)
        for i, gt in enumerate(C.GRAPH_TYPES):
            save(mixed_set(np.random.default_rng(C.SEED + 10 + i), C.N_TRAIN, [gt], label=f"train_{gt}"), f"train_pool_{gt}.pkl")
            save(mixed_set(np.random.default_rng(C.SEED + 20 + i), C.N_VAL, [gt], label=f"val_{gt}"), f"val_pool_{gt}.pkl")

    print("  [3/4] Generating test set...", flush=True)
    test = scenario_set(np.random.default_rng(C.SEED + 2), C.N_NODES,
                        C.TEST_SCENARIOS, C.MISSING_LEVELS, C.TEST_PER_LEVEL)
    save(test, "test.pkl")
    summary(test, "TEST SET")

    if C.RUN_GENERALISATION:
        print("  [4/4] Generating generalisation test sets...", flush=True)
        for n in C.GENERALISE_SIZES:
            gen = scenario_set(np.random.default_rng(C.SEED + 3 + n), n,
                               [C.GENERALISE_SCENARIO], C.GENERALISE_LEVELS, C.TEST_PER_LEVEL)
            save(gen, f"test_n{n}.pkl")

    if C.RUN_BA_EXPERIMENT:
        print("  Generating Barabasi-Albert test set...", flush=True)
        rng = np.random.default_rng(C.SEED + 4)
        ba = []
        for m in C.BA_M_VALUES:
            for k in range(C.BA_PER_M):
                g = make_graph(rng, C.N_NODES, "ba", 0.0, C.NOISE_MODELS["clean"],
                               C.N_ANCHORS, C.COORD_RANGE, ba_m=m)
                g.update(noise="clean", scenario=f"ba/m={m}", ba_m=m, graph_id=k)
                ba.append(g)
            print(f"    BA m={m} generated ({len(ba)} graphs)", flush=True)
        save(ba, "test_ba.pkl")

    print(f"\nSTEP 1 done in {time.time() - t0:.0f}s", flush=True)


In [ ]:
%%writefile step2_train.py
"""
STEP 2 -- train the proposed GNN ONCE on many graphs.

After training, the model is FROZEN. In step 3 it localises every new test
graph with one forward pass (no per-graph training) -- this is what
"the GNN learned the geometry" actually means.

Run:
    python step2_train.py                       # proposed model
    python step2_train.py --variant uniform     # ablation: w_ij = 1 (plain dwMDS steps)
    python step2_train.py --variant huber       # ablation: classical robust weights
    python step2_train.py --variant encoder_only
    python step2_train.py --loss selfsup        # no ground truth at all (stress loss)

Transfer experiment (train on some graph types, test on all in step 3 --transfer):
    python step2_train.py --train-on rgg        # one type
    python step2_train.py --train-on rgg+ba     # two types
    python step2_train.py --train-on all        # all three
"""
import argparse, os, pickle, time, random
import numpy as np
import pandas as pd
import torch
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import config as C
from model import (AnchorGNNLocalizer, collate, supervised_loss, selfsup_stress_loss,
                   to_original_units)
from common import coord_rmse


def load(name):
    with open(os.path.join(C.DATA_DIR, name), "rb") as f:
        return pickle.load(f)


def load_pools(key):
    """N_TRAIN / N_VAL graphs split equally over the types in `key` (same total for every key)."""
    types = C.train_types(key)
    train, val = [], []
    for gt in types:
        train += load(f"train_pool_{gt}.pkl")[:C.N_TRAIN // len(types)]
        val += load(f"val_pool_{gt}.pkl")[:C.N_VAL // len(types)]
    return train, val


def evaluate(model, graphs, device, bs=64, model1=None, dev1=None):
    model.eval(); errs = []
    if model1 is not None and dev1 is not None and len(graphs) >= 2:
        import concurrent.futures
        model1.eval()
        mid = len(graphs) // 2
        g0, g1 = graphs[:mid], graphs[mid:]
        
        def run_chunk(m, chk, dev):
            local_errs = []
            with torch.no_grad():
                for i in range(0, len(chk), bs):
                    b_chk = chk[i:i + bs]
                    B = collate(b_chk, dev)
                    x, _ = m(B, T=None if m.T == 0 else C.TEST_REFINE_STEPS)
                    for g, X in zip(b_chk, to_original_units(B, x)):
                        local_errs.append(coord_rmse(g, X))
            return local_errs
            
        with concurrent.futures.ThreadPoolExecutor(max_workers=2) as pool:
            f0 = pool.submit(run_chunk, model, g0, device)
            f1 = pool.submit(run_chunk, model1, g1, dev1)
            errs = f0.result() + f1.result()
    else:
        with torch.no_grad():
            for i in range(0, len(graphs), bs):
                chunk = graphs[i:i + bs]
                B = collate(chunk, device)
                x, _ = model(B, T=None if model.T == 0 else C.TEST_REFINE_STEPS)
                for g, X in zip(chunk, to_original_units(B, x)):
                    errs.append(coord_rmse(g, X))
    return float(np.nanmean(errs)), float(np.nanmedian(errs))


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--variant", default=C.VARIANT)
    ap.add_argument("--loss", default=C.LOSS_MODE, choices=["supervised", "selfsup"])
    ap.add_argument("--epochs", type=int, default=C.EPOCHS)
    ap.add_argument("--train-on", default=None,
                    help="transfer experiment: graph types to train on, e.g. rgg, rgg+ba, all")
    args = ap.parse_args()

    torch.manual_seed(C.SEED); np.random.seed(C.SEED); random.seed(C.SEED)
    num_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 0
    if num_gpus >= 2:
        print(f"  Dual-GPU acceleration active: utilizing {num_gpus} GPUs (cuda:0, cuda:1)", flush=True)
        for g_idx in range(num_gpus):
            print(f"    GPU {g_idx}: {torch.cuda.get_device_name(g_idx)}", flush=True)
        device = "cuda:0"
        dev0 = torch.device("cuda:0")
        dev1 = torch.device("cuda:1")
        multi_gpu = True
    elif num_gpus == 1:
        device = "cuda:0"
        dev0, dev1 = torch.device("cuda:0"), None
        multi_gpu = False
        print(f"  Single GPU: {torch.cuda.get_device_name(0)}", flush=True)
    else:
        device = "cpu"
        dev0, dev1 = torch.device("cpu"), None
        multi_gpu = False
        print("  CPU  (no GPU found -- on Colab/Kaggle: Enable GPU accelerator)", flush=True)

    tag = args.variant + ("" if args.loss == "supervised" else "_selfsup")
    if args.train_on:
        train, val = load_pools(args.train_on)
        tag += f"__train-{args.train_on}"
    else:
        train, val = load("train.pkl"), load("val.pkl")
    print(f"STEP 2: training '{tag}' on {len(train)} graphs ({'Dual-GPU' if multi_gpu else device})", flush=True)

    # Pre-cache E(2)-canonicalised coordinates and anchor masks to eliminate CPU bottleneck
    for g in train + val:
        if "y_norm" not in g:
            g["y_norm"] = (((g["true_xy"] - g["center"]) / g["scale"]) @ g.get("rot", np.eye(2, dtype=np.float32))).astype(np.float32)
        if "anchor_mask" not in g:
            am = np.zeros(g["n"], bool)
            am[g["anchors"]] = True
            g["anchor_mask"] = am

    model = AnchorGNNLocalizer(in_dim=train[0]["x"].shape[1], edge_dim=train[0]["edge_attr"].shape[1],
                               H=C.HIDDEN, enc_layers=C.ENC_LAYERS, refine_steps=C.REFINE_STEPS,
                               variant=args.variant, huber_delta=C.HUBER_DELTA).to(dev0)
    print(f"  parameters: {sum(p.numel() for p in model.parameters()):,}", flush=True)

    model1 = None
    if multi_gpu:
        import copy
        model1 = copy.deepcopy(model).to(dev1)
        s0 = torch.cuda.Stream(device=dev0)
        s1 = torch.cuda.Stream(device=dev1)

    opt = torch.optim.AdamW(model.parameters(), lr=C.LR, weight_decay=C.WEIGHT_DECAY)
    batch_size = C.BATCH_GRAPHS * 2 if multi_gpu else C.BATCH_GRAPHS
    steps_per_epoch = int(np.ceil(len(train) / batch_size))
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=C.LR, total_steps=args.epochs * steps_per_epoch)
    loss_fn = supervised_loss if args.loss == "supervised" else selfsup_stress_loss

    best, log = np.inf, []
    ckpt = os.path.join(C.CKPT_DIR, f"{tag}.pt")
    for ep in range(1, args.epochs + 1):
        model.train(); random.shuffle(train); t0 = time.time(); tot = 0.0
        for i in range(0, len(train), batch_size):
            b_idx = i // batch_size + 1
            if multi_gpu:
                chunk0 = train[i:i + C.BATCH_GRAPHS]
                chunk1 = train[i + C.BATCH_GRAPHS:i + batch_size]
                B0 = collate(chunk0, dev0)
                if chunk1:
                    B1 = collate(chunk1, dev1)
                    model1.load_state_dict(model.state_dict())
                    with torch.cuda.stream(s0):
                        x0, xA0 = model(B0)
                        loss0 = loss_fn(x0, B0, C.HUBER_DELTA) + C.AUX_WEIGHT * loss_fn(xA0, B0, C.HUBER_DELTA)
                        opt.zero_grad()
                        loss0.backward()
                    with torch.cuda.stream(s1):
                        x1, xA1 = model1(B1)
                        loss1 = loss_fn(x1, B1, C.HUBER_DELTA) + C.AUX_WEIGHT * loss_fn(xA1, B1, C.HUBER_DELTA)
                        model1.zero_grad()
                        loss1.backward()
                    torch.cuda.synchronize(dev0)
                    torch.cuda.synchronize(dev1)
                    for p0, p1 in zip(model.parameters(), model1.parameters()):
                        if p1.grad is not None:
                            p0.grad.add_(p1.grad.to(dev0))
                            p0.grad.mul_(0.5)
                    step_loss = (loss0.item() + loss1.item()) / 2.0
                else:
                    x0, xA0 = model(B0)
                    loss0 = loss_fn(x0, B0, C.HUBER_DELTA) + C.AUX_WEIGHT * loss_fn(xA0, B0, C.HUBER_DELTA)
                    opt.zero_grad()
                    loss0.backward()
                    step_loss = loss0.item()

                torch.nn.utils.clip_grad_norm_(model.parameters(), C.GRAD_CLIP)
                opt.step(); sched.step(); tot += step_loss
            else:
                B = collate(train[i:i + C.BATCH_GRAPHS], device)
                x, xA = model(B)
                loss = loss_fn(x, B, C.HUBER_DELTA) + C.AUX_WEIGHT * loss_fn(xA, B, C.HUBER_DELTA)
                opt.zero_grad(); loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), C.GRAD_CLIP)
                opt.step(); sched.step(); tot += loss.item()
                step_loss = loss.item()

            if b_idx % max(1, steps_per_epoch // 4) == 0 or b_idx == steps_per_epoch:
                print(f"    epoch {ep:2d}/{args.epochs} | batch {b_idx:3d}/{steps_per_epoch} | batch loss: {step_loss:.4f}", flush=True)

        v_mean, v_med = evaluate(model, val, dev0, model1=model1, dev1=dev1)
        log.append(dict(epoch=ep, train_loss=tot / steps_per_epoch, val_rmse_mean=v_mean, val_rmse_median=v_med))
        flag = ""
        if v_mean < best:
            best = v_mean; flag = "  *saved*"
            torch.save(dict(state=model.state_dict(), variant=args.variant, loss=args.loss,
                            train_on=args.train_on,
                            in_dim=train[0]["x"].shape[1], edge_dim=train[0]["edge_attr"].shape[1],
                            H=C.HIDDEN, enc_layers=C.ENC_LAYERS, refine_steps=C.REFINE_STEPS), ckpt)
        print(f"  epoch {ep:3d} | loss {tot / steps_per_epoch:.4f} | val RMSE mean {v_mean:.3f} "
              f"median {v_med:.3f} | {time.time() - t0:.0f}s{flag}", flush=True)

    df = pd.DataFrame(log); df.to_csv(os.path.join(C.RESULTS_DIR, f"train_log_{tag}.csv"), index=False)
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    ax[0].plot(df.epoch, df.train_loss); ax[0].set_yscale("log"); ax[0].set_title("training loss")
    ax[1].plot(df.epoch, df.val_rmse_mean, label="mean"); ax[1].plot(df.epoch, df.val_rmse_median, label="median")
    ax[1].set_title("validation coordinate RMSE"); ax[1].legend()
    for a in ax: a.set_xlabel("epoch"); a.grid(alpha=.3)
    plt.tight_layout()
    plt.savefig(os.path.join(C.RESULTS_DIR, f"train_curve_{tag}.png"), dpi=150)
    plt.savefig(os.path.join(C.RESULTS_DIR, f"train_curve_{tag}.eps"), format="eps")
    plt.close()
    print(f"STEP 2 done. best val RMSE {best:.3f} -> {ckpt}", flush=True)




if __name__ == "__main__":
    main()


In [ ]:
%%writefile step3_evaluate.py
"""
STEP 3 -- compare the proposed GNN with existing methods on the SAME graphs.

For every test graph, every method gets exactly the same inputs
(edges, noisy distances, 3 anchors) and is scored against the same truth.

Outputs (results/):
    all_results.csv              one row per (graph, method)
    summary_table.csv            mean / std / median per method x scenario x missing
    fig_main_rmse.png            average coordinate RMSE vs missing %  (the main figure)
    fig_certified_rmse.png       same, only on nodes whose position is provably unique
    fig_ablation.png             learned vs uniform vs huber vs encoder-only
    fig_runtime.png              seconds per graph
    fig_generalisation.png       trained on n = 30..80, tested on n = 100 / 200
    fig_ba_rigidity.png          Barabasi-Albert m = 1..5 (rigidity threshold)
    fig_examples.png             true vs predicted positions for a few graphs

    transfer_results.csv         (--transfer) each train-on-some-types model on all test graphs
    transfer_matrix.csv          (--transfer) mean RMSE, rows = training set / method, cols = test type
    fig_transfer_matrix.png      (--transfer) the same as a heatmap

Run:  python step3_evaluate.py
      python step3_evaluate.py --plot-only     # only redraw the figures from results/all_results.csv
      python step3_evaluate.py --transfer      # after the main run and step2 --train-on ...
"""
import glob, os, pickle, time, warnings
import numpy as np
import pandas as pd
import torch
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import config as C
from common import coord_rmse, rel_edm_error
from model import AnchorGNNLocalizer, collate, to_original_units
import baselines as BL

warnings.filterwarnings("ignore")
FORCE_CPU = os.environ.get("DEVICE", "").lower() == "cpu"
NUM_GPUS = 0 if FORCE_CPU else (torch.cuda.device_count() if torch.cuda.is_available() else 0)
if NUM_GPUS >= 2:
    DEVICES = [f"cuda:{i}" for i in range(NUM_GPUS)]
    DEVICE = "cuda:0"
elif NUM_GPUS == 1:
    DEVICES = ["cuda:0"]
    DEVICE = "cuda:0"
else:
    DEVICES = ["cpu"]
    DEVICE = "cpu"

OURS = "Ours: AnchorGNN + learned SMACOF"
VARIANT_NAMES = {
    "learned": OURS,
    "uniform": "Ablation: encoder + uniform-weight steps",
    "huber": "Ablation: encoder + Huber-weight steps",
    "encoder_only": "Ablation: encoder only (no refiner)",
    "learned_selfsup": "Ours, self-supervised (no ground truth)",
}


def load(name):
    p = os.path.join(C.DATA_DIR, name)
    if not os.path.exists(p):
        return None
    with open(p, "rb") as f:
        return pickle.load(f)


TRANSFER = "Ours, trained on "          # + "rgg", "rgg+ba", "all", ...


def load_models(transfer=False):
    """Main comparison: every checkpoint except the transfer ones.
    transfer=True: only the learned-variant transfer checkpoints (step2 --train-on)."""
    models = {}
    for path in sorted(glob.glob(os.path.join(C.CKPT_DIR, "*.pt"))):
        tag = os.path.basename(path)[:-3]
        if ("__train-" in tag) != transfer:
            continue
        ck = torch.load(path, map_location=DEVICE, weights_only=False)
        if transfer and ck["variant"] != "learned":
            continue
        m = AnchorGNNLocalizer(ck["in_dim"], ck["edge_dim"], ck["H"], ck["enc_layers"],
                               ck["refine_steps"], ck["variant"], C.HUBER_DELTA).to(DEVICE)
        m.load_state_dict(ck["state"]); m.eval()
        name = TRANSFER + ck["train_on"] if transfer else VARIANT_NAMES.get(tag, tag)
        models[name] = m
    return models


def run_gnn(model, graphs, bs=64):
    """One forward pass per batch; parallel across available GPUs."""
    import copy, concurrent.futures
    if len(DEVICES) >= 2 and len(graphs) >= 2:
        models_per_dev = [model if dev == DEVICE else copy.deepcopy(model).to(dev).eval()
                          for dev in DEVICES]
        split_indices = np.array_split(np.arange(len(graphs)), len(DEVICES))
        
        def run_dev(m, chk, dev):
            out, t = [], 0.0
            with torch.no_grad():
                for i in range(0, len(chk), bs):
                    b_chk = chk[i:i + bs]
                    B = collate(b_chk, dev)
                    t0 = time.perf_counter(); x, _ = m(B, T=None if m.T == 0 else C.TEST_REFINE_STEPS)
                    if "cuda" in dev: torch.cuda.synchronize(dev)
                    t += time.perf_counter() - t0
                    out += to_original_units(B, x)
            return out, t

        with concurrent.futures.ThreadPoolExecutor(max_workers=len(DEVICES)) as pool:
            futures = [pool.submit(run_dev, models_per_dev[idx], [graphs[k] for k in split_indices[idx]], DEVICES[idx])
                       for idx in range(len(DEVICES))]
            all_out = []
            total_t = 0.0
            for f in futures:
                o, t_dev = f.result()
                all_out.extend(o)
                total_t += t_dev
        return all_out, total_t / len(graphs)
    else:
        out, t = [], 0.0
        with torch.no_grad():
            for i in range(0, len(graphs), bs):
                chunk = graphs[i:i + bs]
                B = collate(chunk, DEVICE)
                t0 = time.perf_counter(); x, _ = model(B, T=None if model.T == 0 else C.TEST_REFINE_STEPS)
                if DEVICE.startswith("cuda"): torch.cuda.synchronize()
                t += time.perf_counter() - t0
                out += to_original_units(B, x)
        return out, t / len(graphs)


def score_rows(g, method, X, secs):
    return dict(scenario=g["scenario"], missing=int(round(100 * g["missing_nominal"])), n=g["n"],
                graph_id=g["graph_id"], method=method, rmse=coord_rmse(g, X),
                rmse_certified=coord_rmse(g, X, g["certified"]), rel_edm=rel_edm_error(g, X),
                seconds=secs, globally_rigid=bool(g["globally_rigid"]),
                frac_certified=float(g["certified"].mean()), ba_m=g.get("ba_m"))


def warmup_cuda(devices):
    for dev in devices:
        if "cuda" in dev:
            try:
                dummy = torch.eye(4, device=dev)
                _ = torch.linalg.eigh(dummy)
                _ = torch.linalg.svd(dummy)
            except Exception:
                pass


def evaluate_set(graphs, models, baselines, polish=True, label=""):
    import concurrent.futures, threading
    rows, preds = [], {}
    n_graphs = len(graphs)
    
    # Warm up cuSOLVER sequentially on all GPUs to prevent lazy wrapper race condition
    warmup_cuda(DEVICES)

    # Pre-build baselines per device for round-robin GPU dispatch
    base_per_dev = {dev: BL.get_baselines(C, device=dev) for dev in DEVICES}
    gpu_methods = {
        "DV-distance + dwMDS (no learning)", "Classical MDS (direct)",
        "Rank Alternation (EDM book Alg 2)", "OptSpace (EDM book Alg 3)",
        "Alternating Descent (EDM book Alg 4)", "Yan et al. GCN (3 anchor labels)",
        "Per-graph GraphSAGE (previous)"
    }

    # 1. Models
    for name, m in models.items():
        print(f"\n{label} [Model] Running '{name}' on {n_graphs} test graphs ({'Dual-GPU' if len(DEVICES) >= 2 else DEVICE})...", flush=True)
        P, sec = run_gnn(m, graphs)
        preds[name] = P
        rows += [score_rows(g, name, X, sec) for g, X in zip(graphs, P)]
        m_rmse = np.nanmean([coord_rmse(g, X) for g, X in zip(graphs, P)])
        print(f"{label} [Model] '{name}' done -> Mean RMSE: {m_rmse:.4f} ({sec*1000:.1f} ms/graph)", flush=True)

        if polish and (name == OURS or name.startswith(TRANSFER)):
            n_cpu_workers = min(os.cpu_count() or 4, 8)
            print(f"{label} [Polish] Running Huber L-BFGS polish on {n_graphs} graphs (parallel across {n_cpu_workers} CPU workers)...", flush=True)
            t_pol_start = time.time()
            log_step = max(1, n_graphs // 5)
            
            def polish_one(item):
                idx, g, X = item
                t0 = time.perf_counter()
                Y = BL.stress_lbfgs(g, X, robust=True)
                dur = time.perf_counter() - t0
                return idx, Y, dur

            P2 = [None] * n_graphs
            polish_rows = [None] * n_graphs
            completed = 0
            print_lock = threading.Lock()

            with concurrent.futures.ThreadPoolExecutor(max_workers=n_cpu_workers) as pool:
                futures = [pool.submit(polish_one, (idx, g, X)) for idx, (g, X) in enumerate(zip(graphs, P))]
                for f in concurrent.futures.as_completed(futures):
                    idx, Y, dur = f.result()
                    P2[idx] = Y
                    polish_rows[idx] = score_rows(graphs[idx], name + " + Huber polish", Y, sec + dur)
                    with print_lock:
                        completed += 1
                        if completed % log_step == 0 or completed == n_graphs:
                            elapsed = time.time() - t_pol_start
                            eta = (elapsed / completed) * (n_graphs - completed)
                            print(f"    progress: {completed}/{n_graphs} polished ({completed*100//n_graphs}%) | elapsed: {elapsed:.1f}s | ETA: {eta:.1f}s", flush=True)

            rows.extend(polish_rows)
            preds[name + " + Huber polish"] = P2
            pol_rmse = np.nanmean([coord_rmse(g, Y) for g, Y in zip(graphs, P2)])
            print(f"{label} [Polish] Finished in {time.time() - t_pol_start:.1f}s -> Mean RMSE: {pol_rmse:.4f}", flush=True)


    # 2. Baselines
    n_base = len(baselines)
    for b_idx, (name, fn) in enumerate(baselines.items(), 1):
        is_gpu = (name in gpu_methods) and (DEVICES[0] != "cpu")
        if is_gpu and len(DEVICES) >= 2:
            worker_count = len(DEVICES)
            mode_str = f"Dual-GPU parallel ({', '.join(DEVICES)})"
        elif is_gpu:
            worker_count = 1
            mode_str = DEVICE
        else:
            worker_count = min(os.cpu_count() or 4, 8)
            mode_str = f"Multi-core CPU ({worker_count} workers)"

        print(f"\n{label} [{b_idx}/{n_base}] Baseline: '{name}' on {n_graphs} graphs [{mode_str}]...", flush=True)
        t0 = time.time()
        P = [None] * n_graphs
        base_rows = [None] * n_graphs
        log_step = max(1, n_graphs // 5)
        completed = 0
        print_lock = threading.Lock()

        def run_one_baseline(item):
            k, g = item
            dev = DEVICES[k % len(DEVICES)] if is_gpu else DEVICE
            target_fn = base_per_dev[dev].get(name, fn)
            X, sec = BL.timed(target_fn, g)
            return k, X, sec

        with concurrent.futures.ThreadPoolExecutor(max_workers=worker_count) as pool:
            futures = [pool.submit(run_one_baseline, (k, g)) for k, g in enumerate(graphs)]
            for f in concurrent.futures.as_completed(futures):
                k, X, sec = f.result()
                P[k] = X
                base_rows[k] = score_rows(graphs[k], name, X, sec)
                with print_lock:
                    completed += 1
                    if completed % log_step == 0 or completed == n_graphs:
                        elapsed = time.time() - t0
                        eta = (elapsed / completed) * (n_graphs - completed)
                        print(f"    progress: {completed}/{n_graphs} graphs ({completed*100//n_graphs}%) | elapsed: {elapsed:.1f}s | ETA: {eta:.1f}s", flush=True)

        preds[name] = P
        rows.extend(base_rows)
        b_rmse = np.nanmean([coord_rmse(g, X) for g, X in zip(graphs, P)])
        print(f"{label} [{b_idx}/{n_base}] '{name}' completed in {time.time() - t0:.1f}s -> Mean RMSE: {b_rmse:.4f}", flush=True)

    return pd.DataFrame(rows), preds




# -----------------------------------------------------------------------------
# plotting
# -----------------------------------------------------------------------------
STYLE = {OURS: dict(color="crimson", lw=2.6, marker="o"),
         OURS + " + Huber polish": dict(color="darkred", lw=2.0, marker="s", ls="--")}


PALETTE = ["#1f77b4", "#2ca02c", "#ff7f0e", "#9467bd", "#8c564b", "#e377c2", "#7f7f7f",
           "#17becf", "#bcbd22", "#393b79"]
FLOOR = 1e-6     # errors below this are numerically exact; drawn on the floor


def style_for(mth, methods):
    if mth in STYLE:
        return STYLE[mth]
    others = [m for m in methods if m not in STYLE]
    return dict(color=PALETTE[others.index(mth) % len(PALETTE)], lw=1.3, marker=".")


def save_fig_both(fname, dpi=150):
    p_png = os.path.join(C.RESULTS_DIR, fname)
    p_eps = os.path.join(C.RESULTS_DIR, fname.replace(".png", ".eps"))
    plt.savefig(p_png, dpi=dpi)
    plt.savefig(p_eps, format="eps")
    plt.close()


def plot_vs_missing(df, metric, methods, fname, title, ylabel):
    """Line = MEAN over the test graphs at each missing level.
    Shaded band = 25th-75th percentile (a +/- std band is meaningless on a log axis)."""
    scen = list(dict.fromkeys(df.scenario))
    cols = 2; rows = int(np.ceil(len(scen) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(7 * cols, 4.6 * rows), squeeze=False)
    for ax, sc in zip(axes.ravel(), scen):
        d = df[df.scenario == sc]
        for mth in methods:
            g = d[d.method == mth].groupby("missing")[metric]
            if g.ngroups == 0: continue
            mean, q1, q3 = g.mean(), g.quantile(0.25), g.quantile(0.75)
            st = style_for(mth, methods)
            ax.plot(mean.index, np.maximum(mean, FLOOR), label=mth, **st)
            ax.fill_between(mean.index, np.maximum(q1, FLOOR), np.maximum(q3, FLOOR),
                            color=st.get("color"), alpha=0.10, lw=0)
        ax.set_yscale("log"); ax.set_ylim(bottom=FLOOR * 0.7); ax.set_title(sc)
        ax.set_xlim(df.missing.min() - 3, df.missing.max() + 3)   # same x range in every panel (BA starts at 50 %)
        ax.set_xlabel("missing distances (%)")
        ax.set_ylabel(f"{ylabel}  (floor = exact)"); ax.grid(alpha=.3, which="both")
    for ax in axes.ravel()[len(scen):]: ax.axis("off")
    h, l = axes[0, 0].get_legend_handles_labels()
    fig.legend(h, l, loc="lower center", ncol=3, fontsize=8, frameon=False)
    fig.suptitle(title + "\n(line = mean over test graphs, band = 25-75th percentile)"); plt.tight_layout(rect=(0, 0.08, 1, 0.95))
    save_fig_both(fname, dpi=150)


def plot_examples(graphs, preds, methods, fname, levels=(30, 70, 90)):
    pick = []
    for lv in levels:
        for i, g in enumerate(graphs):
            if int(round(100 * g["missing_nominal"])) == lv:
                pick.append(i); break
    if not pick: return
    fig, axes = plt.subplots(len(methods), len(pick), figsize=(4.2 * len(pick), 4 * len(methods)), squeeze=False)
    for c, i in enumerate(pick):
        g = graphs[i]; T = g["true_xy"]
        for r, mth in enumerate(methods):
            ax = axes[r, c]; X = preds[mth][i]
            for a, b in g["edges"]:
                ax.plot(T[[a, b], 0], T[[a, b], 1], c="0.85", lw=0.4, zorder=0)
            ax.scatter(T[:, 0], T[:, 1], s=40, facecolors="none", edgecolors="tab:blue", label="true")
            ax.scatter(X[:, 0], X[:, 1], s=12, c="tab:red", label="predicted")
            for k in range(g["n"]):
                ax.plot([T[k, 0], X[k, 0]], [T[k, 1], X[k, 1]], c="gray", lw=0.5)
            ax.scatter(*g["anchor_xy"].T, marker="*", s=180, c="green", zorder=5, label="anchors")
            ax.set_title(f"{mth[:34]}\n{g['scenario']} {int(100 * g['missing_nominal'])}% missing, "
                         f"RMSE {coord_rmse(g, X):.2f}", fontsize=8)
            ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])
    axes[0, 0].legend(fontsize=7)
    plt.tight_layout()
    save_fig_both(fname, dpi=130)


def main_plots(df):
    """All figures that only need all_results.csv (re-drawable with --plot-only)."""
    ablations = [m for m in df.method.unique() if m.startswith("Ablation") or m.startswith("Ours, self")]
    main_methods = [m for m in df.method.unique() if m not in ablations]
    plot_vs_missing(df, "rmse", main_methods, "fig_main_rmse.png",
                    "Average coordinate RMSE on identical test graphs", "coordinate RMSE")
    plot_vs_missing(df, "rmse_certified", main_methods, "fig_certified_rmse.png",
                    "RMSE on nodes whose position is provably unique", "certified-node RMSE")
    plot_vs_missing(df, "rel_edm", main_methods, "fig_relative_error.png",
                    "Average Relative EDM Error ||D - D_hat||_F / ||D||_F (Dokmanic et al.)", "relative EDM error")
    if ablations:
        plot_vs_missing(df, "rmse", [OURS] + ablations + ["DV-distance + dwMDS (no learning)"],
                        "fig_ablation.png", "Ablation: what does each part contribute?", "coordinate RMSE")
    rt = df.groupby("method").seconds.mean().sort_values()
    plt.figure(figsize=(8, 0.45 * len(rt) + 1)); plt.barh(rt.index, rt.values, color="slategray")
    plt.xscale("log"); plt.xlabel("seconds per graph (log)"); plt.title("Runtime per graph")
    plt.tight_layout()
    save_fig_both("fig_runtime.png", dpi=150)



# -----------------------------------------------------------------------------
# transfer experiment: train on some graph types, test on all
# -----------------------------------------------------------------------------
SEQ_BLUE = ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"]


def transfer_matrix(df):
    """Mean RMSE over missing >= TRANSFER_MIN_MISSING: rows = method, cols = test scenario."""
    d = df[df.missing >= 100 * C.TRANSFER_MIN_MISSING]
    cols = [f"{gt}/{nz}" for gt in C.GRAPH_TYPES for nz in C.NOISE_MODELS if f"{gt}/{nz}" in set(d.scenario)]
    piv = d.pivot_table(index="method", columns="scenario", values="rmse", aggfunc="mean")[cols]
    ours = [TRANSFER + k for k in C.TRANSFER_TRAIN_SETS]
    blocks = [[m for m in ours if m in piv.index],
              [m + " + Huber polish" for m in ours if m + " + Huber polish" in piv.index],
              [m for m in dict.fromkeys(df.method) if not m.startswith(TRANSFER) and m in piv.index]]
    return piv.loc[sum(blocks, [])], blocks


def plot_transfer(piv, blocks, fname):
    from matplotlib.colors import LinearSegmentedColormap, LogNorm
    from matplotlib.patches import Rectangle
    cmap = LinearSegmentedColormap.from_list("seq_blue", SEQ_BLUE)
    norm = LogNorm(vmin=1e-3, vmax=max(piv.values.max(), 1e-2))
    V = piv.values
    fig, ax = plt.subplots(figsize=(2.0 * V.shape[1] + 5, 0.42 * V.shape[0] + 2.2))
    ax.imshow(np.clip(V, 1e-3, None), cmap=cmap, norm=norm, aspect="auto")
    best = np.nanargmin(V, axis=0)
    for i, mth in enumerate(piv.index):
        seen = C.train_types(mth[len(TRANSFER):].replace(" + Huber polish", "")) if mth.startswith(TRANSFER) else None
        for j, sc in enumerate(piv.columns):
            v = V[i, j]
            txt = "<0.001" if v < 1e-3 else f"{v:.3f}" if v < 1 else f"{v:.2f}"
            dark = norm(max(v, 1e-3)) > 0.55
            ax.text(j, i, txt, ha="center", va="center", fontsize=8,
                    color="white" if dark else "#1a1a1a", fontweight="bold" if best[j] == i else "normal")
            if seen is not None and sc.split("/")[0] not in seen:      # graph type never seen in training
                ax.add_patch(Rectangle((j - 0.46, i - 0.42), 0.92, 0.84, fill=False, ls="--", lw=1.2,
                                       ec="white" if dark else "#1a1a1a"))
    edge = 0
    for b in blocks[:-1]:                     # white gaps between the three row blocks
        edge += len(b)
        if 0 < edge < len(piv.index): ax.axhline(edge - 0.5, color="white", lw=4)
    ax.set_xticks(range(V.shape[1])); ax.set_xticklabels(piv.columns, fontsize=9)
    ax.set_yticks(range(V.shape[0])); ax.set_yticklabels(piv.index, fontsize=8)
    ax.xaxis.tick_top(); ax.tick_params(length=0)
    for s in ax.spines.values(): s.set_visible(False)
    ax.set_title(f"Mean coordinate RMSE, missing >= {int(100 * C.TRANSFER_MIN_MISSING)} %  (test graph type / noise; "
                 "random = spanning tree + random edges)\n"
                 "dashed box = graph type never seen in training  ·  bold = best in column  ·  darker = worse",
                 fontsize=9, pad=28)
    plt.tight_layout(); save_fig_both(fname, dpi=150)


def run_transfer(plot_only=False):
    out = os.path.join(C.RESULTS_DIR, "transfer_results.csv")
    if plot_only:
        df = pd.read_csv(out)
    else:
        models = load_models(transfer=True)
        if not models:
            raise SystemExit("No transfer checkpoints -- run  python step2_train.py --train-on <types>  first.")
        main_csv, test_pkl = os.path.join(C.RESULTS_DIR, "all_results.csv"), os.path.join(C.DATA_DIR, "test.pkl")
        if not os.path.exists(main_csv) or os.path.getmtime(main_csv) < os.path.getmtime(test_pkl):
            raise SystemExit("Run  python step3_evaluate.py  (main comparison) first on the current test set: "
                             "the transfer table reuses its existing-method rows.")
        print(f"STEP 3 (transfer): {list(models)}")
        df, _ = evaluate_set(load("test.pkl"), models, {}, label="[transfer]")
        base = pd.read_csv(main_csv)
        base = base[~base.method.str.startswith(("Ours", "Ablation"))]   # existing methods: no training, same graphs
        df = pd.concat([df, base], ignore_index=True)
        df.to_csv(out, index=False)
    piv, blocks = transfer_matrix(df)
    piv.to_csv(os.path.join(C.RESULTS_DIR, "transfer_matrix.csv"))
    plot_transfer(piv, blocks, "fig_transfer_matrix.png")
    print(f"\nTRANSFER MATRIX: mean RMSE, missing >= {int(100 * C.TRANSFER_MIN_MISSING)} %\n", piv.round(3).to_string())


def main():
    import sys
    if "--transfer" in sys.argv:
        run_transfer(plot_only="--plot-only" in sys.argv); return
    if "--plot-only" in sys.argv:          # redraw figures from saved results, no re-running
        main_plots(pd.read_csv(os.path.join(C.RESULTS_DIR, "all_results.csv")))
        print("figures redrawn"); return
    mode_info = f"Dual-GPU mode ({', '.join(DEVICES)})" if len(DEVICES) >= 2 else f"Device: {DEVICE}"
    print(f"STEP 3: evaluation on {mode_info}", flush=True)
    models = load_models()
    if not models:
        raise SystemExit("No checkpoints found -- run step2_train.py first.")
    print("  models:", list(models), flush=True)
    baselines = BL.get_baselines(C, device=DEVICE)
    test = load("test.pkl")


    # ---------------- main comparison ----------------
    df, preds = evaluate_set(test, models, baselines, label="[main]")
    df.to_csv(os.path.join(C.RESULTS_DIR, "all_results.csv"), index=False)
    summ = (df.groupby(["scenario", "missing", "method"])
              .agg(rmse_mean=("rmse", "mean"), rmse_std=("rmse", "std"), rmse_median=("rmse", "median"),
                   rmse_cert_mean=("rmse_certified", "mean"), rel_edm_mean=("rel_edm", "mean"),
                   sec_per_graph=("seconds", "mean"), frac_rigid=("globally_rigid", "mean"))
              .reset_index())
    summ.to_csv(os.path.join(C.RESULTS_DIR, "summary_table.csv"), index=False)

    main_plots(df)

    show = [m for m in [OURS, "MDS-MAP + Huber stress", "SDP (Biswas-Ye)", "Yan et al. GCN (3 anchor labels)", "MPNN-Loc (Xu & Lin 2025)"]
            if m in preds]
    plot_examples(test, preds, show, "fig_examples.png")

    # ---------------- printed table ----------------
    piv = summ.pivot_table(index=["scenario", "missing"], columns="method", values="rmse_mean")
    print("\nMEAN COORDINATE RMSE (lower is better)\n", piv.round(3).to_string())

    # ---------------- generalisation to bigger graphs ----------------
    fast = {k: v for k, v in baselines.items() if "Per-graph" not in k and "Yan" not in k and "MPNN-Loc" not in k}
    gen_rows = []
    for n in C.GENERALISE_SIZES:
        G = load(f"test_n{n}.pkl")
        if G is None: continue
        fb = {k: v for k, v in fast.items() if not (k.startswith("SDP") and n > C.SDP_MAX_N)}
        d, _ = evaluate_set(G, {OURS: models[OURS]} if OURS in models else models, fb, label=f"[n={n}]")
        gen_rows.append(d)
    if gen_rows:
        gd = pd.concat(gen_rows); gd.to_csv(os.path.join(C.RESULTS_DIR, "generalisation.csv"), index=False)
        fig, axes = plt.subplots(1, len(C.GENERALISE_SIZES), figsize=(7 * len(C.GENERALISE_SIZES), 4.5), squeeze=False)
        for ax, n in zip(axes[0], C.GENERALISE_SIZES):
            d = gd[gd.n == n]
            for mth in d.method.unique():
                s = d[d.method == mth].groupby("missing").rmse.mean()
                ax.plot(s.index, s.values, label=mth, **style_for(mth, list(d.method.unique())))
            ax.set_yscale("log"); ax.set_title(f"n = {n} (never seen in training)")
            ax.set_xlabel("missing (%)"); ax.set_ylabel("coordinate RMSE"); ax.grid(alpha=.3)
        axes[0, 0].legend(fontsize=7); plt.tight_layout()
        save_fig_both("fig_generalisation.png", dpi=150)

    # ---------------- Barabasi-Albert rigidity threshold ----------------
    ba = load("test_ba.pkl")
    if ba is not None:
        d, _ = evaluate_set(ba, {OURS: models[OURS]} if OURS in models else models, fast,
                            polish=False, label="[BA]")
        d.to_csv(os.path.join(C.RESULTS_DIR, "ba_results.csv"), index=False)
        fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
        for mth in d.method.unique():
            s = d[d.method == mth].groupby("ba_m").rmse.mean()
            ax[0].plot(s.index, s.values, label=mth, **style_for(mth, list(d.method.unique())))
        ax[0].set_yscale("log"); ax[0].set_xlabel("BA parameter m (links per new node)")
        ax[0].set_ylabel("coordinate RMSE"); ax[0].legend(fontsize=7); ax[0].grid(alpha=.3)
        r = d.drop_duplicates(["ba_m", "graph_id"]).groupby("ba_m")[["globally_rigid", "frac_certified"]].mean()
        ax[1].bar(r.index - 0.2, r.globally_rigid, 0.4, label="graphs globally rigid")
        ax[1].bar(r.index + 0.2, r.frac_certified, 0.4, label="nodes certified unique")
        ax[1].set_xlabel("m"); ax[1].set_ylim(0, 1.05); ax[1].legend(); ax[1].set_title("Is the answer unique?")
        plt.tight_layout(); save_fig_both("fig_ba_rigidity.png", dpi=150)

    print(f"\nSTEP 3 done. Figures and CSVs in {C.RESULTS_DIR}")


if __name__ == "__main__":
    main()


## Step 1 — build all datasets
Main test set (spanning tree / RGG / BA), n = 100/200, BA m = 1…5, per-type pools for the transfer experiment.

In [ ]:
sh("python step1_data.py")

## Step 2 — train
A model whose checkpoint already exists is skipped (useful after a disconnect with `USE_DRIVE = True`).

In [ ]:
def train(args, tag):
    if os.path.exists(f"checkpoints/{tag}.pt"):
        print(f"skip {tag}: checkpoint exists"); return
    print(f"\n===== training {tag} =====", flush=True)
    sh(f"python step2_train.py {args}")

train("", "learned")                                          # proposed model
if RUN_ABLATIONS:
    for v in ["uniform", "huber", "encoder_only"]:
        train(f"--variant {v}", v)

## Step 3 — compare all methods on identical test graphs

In [ ]:
sh("python step3_evaluate.py")

## Transfer across graph types
Train on 1, 2 or all 3 graph types; test every model on all three. Same training-set size for every model.

In [ ]:
if RUN_TRANSFER:
    for k in ["random", "rgg", "ba", "random+rgg", "random+ba", "rgg+ba", "all"]:
        train(f"--train-on {k}", f"learned__train-{k}")
    sh("python step3_evaluate.py --transfer")
else:
    print("RUN_TRANSFER = False: skipped")

## Results

In [ ]:
import glob, pandas as pd
from IPython.display import Image, display
for f in ["fig_main_rmse", "fig_certified_rmse", "fig_ablation", "fig_transfer_matrix", "fig_runtime",
          "fig_generalisation", "fig_ba_rigidity", "fig_examples"]:
    p = f"results/{f}.png"
    if os.path.exists(p):
        print(f); display(Image(p))
for p in sorted(glob.glob("results/train_curve_*.png")):
    print(os.path.basename(p)); display(Image(p))

In [ ]:
summ = pd.read_csv("results/summary_table.csv")
piv = summ.pivot_table(index=["scenario", "missing"], columns="method", values="rmse_mean")
piv.round(3)

In [ ]:
if os.path.exists("results/transfer_matrix.csv"):
    display(pd.read_csv("results/transfer_matrix.csv", index_col=0).round(3))

## Optional — fair runtime (everything on CPU)
Our model ran on the GPU above, the existing methods on CPU. For the runtime numbers in the paper, re-run step 3 on CPU. The GPU results are kept in `results_gpu/`.

In [ ]:
RUN_CPU_TIMING = False
if RUN_CPU_TIMING:
    sh("rm -rf results_gpu && cp -r results results_gpu")
    sh("python step3_evaluate.py", DEVICE="cpu")
    print(pd.read_csv("results/summary_table.csv").groupby("method").sec_per_graph.mean().sort_values())

## Download everything

In [ ]:
!zip -qr gnn_loc_results.zip results results_gpu checkpoints 2>/dev/null; ls -lh gnn_loc_results.zip
try:
    from google.colab import files
    files.download("gnn_loc_results.zip")
except ImportError:
    import shutil
    if os.path.exists("/kaggle/working") and not os.path.exists("/kaggle/working/gnn_loc_results.zip"):
        shutil.copy("gnn_loc_results.zip", "/kaggle/working/gnn_loc_results.zip")
    print("\n[+] Results zipped successfully!\nIn Kaggle: Look at the right sidebar under 'Output' (or /kaggle/working/gnn_loc_results.zip) to download.")
